# LeetCode 315 — Count of Smaller Numbers After Self (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countSmaller(self, nums: List[int]) -> List[int]:

```

Đề bài rất ngắn: với **mỗi** phần tử của mảng, hãy đếm xem **bên phải nó** có bao nhiêu số **nhỏ hơn nó**. Người mới nào cũng nghĩ ra ngay cách "với mỗi phần tử, nhìn hết sang bên phải và đếm" — và cách đó **đúng**. Bài được xếp Hard vì với `n = 10^5` phần tử, cách đó phải làm khoảng `5·10^9` phép so sánh → quá chậm. Ta cần một cách **đếm cả nhóm một lúc**.

Bài này là "sân tập" tuyệt vời cho **ba kỹ thuật kinh điển** mà bạn sẽ gặp lại rất nhiều lần, và notebook sẽ dạy cả ba **từ con số 0**:

1. **Tìm kiếm nhị phân (binary search)** với module `bisect` của Python.
2. **Merge sort (sắp xếp trộn)** — và mẹo **"đếm trong lúc trộn"**.
3. **Fenwick Tree (Binary Indexed Tree — BIT)** kết hợp **nén tọa độ (coordinate compression)**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đề, **tính tay** đáp án cho từng phần tử và hiểu hai chữ quan trọng: "**nhỏ hơn hẳn**" (số bằng nhau không tính) và "**bên phải**".
2. **Cách 1 — Brute force** hai vòng lặp `O(n²)`: dùng làm **"trọng tài"** kiểm tra mọi cách khác.
3. **Đổi chiều duyệt: từ phải sang trái** với một "túi" chứa các số đã gặp — khung chung của nhiều lời giải nhanh.
4. **Cách 2 — List sắp xếp + `bisect`**: học binary search từ đầu, tự cài `bisect_left`, hiểu vì sao `insort` tốn `O(n)` mỗi lần (dù chạy nhanh trong thực tế).
5. **Cách 3 — Merge sort trên mảng chỉ số** (cách chính, `O(n log n)`): học đệ quy, trộn hai list đã sắp xếp, mẹo đếm các phần tử bên phải đã **"vượt lên"**; trace kỹ trên `[5, 2, 6, 1]`.
6. **Cách 4 — Fenwick Tree + nén tọa độ**: dạy mảng đếm, nén tọa độ, BIT từ con số `i & -i`; trace từng bước.
7. **Cách 5 (tùy chọn) — Cây nhị phân tìm kiếm (BST) tự cài**, mỗi nút đếm kích thước cây con trái.
8. Xử lý **số trùng** và **số âm**, code nộp LeetCode, test đối chiếu chéo cả 5 cách, vì sao đúng, so sánh, lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Cho một mảng số nguyên `nums`, hãy trả về một mảng số nguyên `counts`, trong đó **`counts[i]` là số lượng phần tử nhỏ hơn `nums[i]` nằm ở bên phải `nums[i]`**.

**Example 1**

```text
Input:  nums = [5,2,6,1]
Output: [2,1,1,0]
Explanation:
To the right of 5 there are 2 smaller elements (2 and 1).
To the right of 2 there is only 1 smaller element (1).
To the right of 6 there is 1 smaller element (1).
To the right of 1 there is 0 smaller element.
```

**Example 2**

```text
Input:  nums = [-1]
Output: [0]
```

**Example 3**

```text
Input:  nums = [-1,-1]
Output: [0,0]
```

**Constraints**

- `1 <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`

Ba chi tiết cần nhớ ngay từ đầu:

- "**Nhỏ hơn**" nghĩa là **nhỏ hơn hẳn** (`<`), không phải `<=`. Example 3 cho thấy: hai số `-1` bằng nhau nên **không** đếm nhau → `[0, 0]`.
- "**Bên phải**" nghĩa là chỉ xét các chỉ số `j > i`. Các số đứng bên trái không liên quan.
- Mảng **có thể có số âm** (tới `-10^4`) và **số trùng nhau**. Hai điều này sẽ ảnh hưởng tới cách cài đặt ở Cách 2, 3 và 4.

## 2. Hiểu đề bằng tay: tính từng `counts[i]` của Example 1

`nums = [5, 2, 6, 1]`. Với mỗi chỉ số `i`, ta nhìn phần **bên phải** `nums[i+1:]` rồi đếm số nào `< nums[i]`:

| i | `nums[i]` | Bên phải `nums[i+1:]` | Các số nhỏ hơn | `counts[i]` |
|:---:|:---:|:---|:---|:---:|
| 0 | 5 | `[2, 6, 1]` | 2, 1 | **2** |
| 1 | 2 | `[6, 1]` | 1 | **1** |
| 2 | 6 | `[1]` | 1 | **1** |
| 3 | 1 | `[]` | (không có) | **0** |

→ `counts = [2, 1, 1, 0]`, khớp Output của đề.

**Python cần biết cho cell sau:**

- **Index (chỉ số)** bắt đầu từ `0`: `nums[0]` là `5`, `nums[3]` là `1`.
- `nums[i+1:]` là **slicing** (cắt list): lấy từ chỉ số `i+1` tới **hết** list. Khi `i` là chỉ số cuối, `nums[i+1:]` là list rỗng `[]` — không lỗi.
- `[y for y in right if y < x]` là **list comprehension**: "tạo list mới gồm các `y` trong `right` thỏa `y < x`".
- `len(...)` trả về số phần tử của list.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

examples = [
    ([5, 2, 6, 1], [2, 1, 1, 0]),   # (nums, expected)
    ([-1], [0]),
    ([-1, -1], [0, 0]),
]

nums = [5, 2, 6, 1]
for i in range(len(nums)):
    x = nums[i]
    right = nums[i + 1:]                      # phần bên phải của nums[i]
    smaller = [y for y in right if y < x]     # các số nhỏ hơn HẲN x
    print(f"i={i}, nums[i]={x}, bên phải={right}, nhỏ hơn={smaller} -> counts[{i}] = {len(smaller)}")

i=0, nums[i]=5, bên phải=[2, 6, 1], nhỏ hơn=[2, 1] -> counts[0] = 2
i=1, nums[i]=2, bên phải=[6, 1], nhỏ hơn=[1] -> counts[1] = 1
i=2, nums[i]=6, bên phải=[1], nhỏ hơn=[1] -> counts[2] = 1
i=3, nums[i]=1, bên phải=[], nhỏ hơn=[] -> counts[3] = 0


### Hai quan sát quan trọng trước khi giải

**Quan sát 1 — Số trùng không được đếm.** Với `nums = [2, 2, 1]`: bên phải số `2` đầu tiên có `[2, 1]`, nhưng chỉ số `1` nhỏ hơn hẳn → `counts[0] = 1`, **không phải 2**. Mọi lời giải nhanh phía sau đều phải cẩn thận ở điểm này (chọn `bisect_left` chứ không phải `bisect_right`, chọn `<` hay `<=` khi trộn, `query(r - 1)` chứ không phải `query(r)`...).

**Quan sát 2 — Đây là bài "đếm cặp nghịch thế" (inversion) chia theo từng phần tử.** Một cặp chỉ số `(i, j)` gọi là **nghịch thế (inversion)** nếu `i < j` nhưng `nums[i] > nums[j]` — tức là hai số "đứng sai thứ tự" so với khi sắp xếp tăng dần. `counts[i]` chính là số nghịch thế mà `i` đứng ở vị trí bên trái. Cộng tất cả `counts` lại ta được **tổng số nghịch thế** của mảng. Đây là gợi ý lớn: thuật toán đếm nghịch thế nổi tiếng nhất là **merge sort** — ta sẽ dùng nó ở Cách 3.

**Quan sát 3 — Số cặp cần xét rất lớn.** Mảng dài `n` có `n(n−1)/2` cặp `(i, j)` với `i < j`. Với `n = 10^5` là gần **5 tỉ** cặp.

In [2]:
def count_smaller_by_definition(nums):
    """Dịch thẳng định nghĩa bằng list comprehension (chỉ dùng để minh họa)."""
    return [len([y for y in nums[i + 1:] if y < nums[i]]) for i in range(len(nums))]

print("[2, 2, 1]    ->", count_smaller_by_definition([2, 2, 1]), " (số 2 không đếm số 2)")
print("[-1, -1]     ->", count_smaller_by_definition([-1, -1]))

nums = [5, 2, 6, 1]
inversions = [(i, j) for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] > nums[j]]
print("Các cặp nghịch thế của [5, 2, 6, 1]:", inversions)
print("Số nghịch thế =", len(inversions), "= tổng counts =", sum(count_smaller_by_definition(nums)))

for n in [4, 1000, 10**5]:
    print(f"n = {n:>6}: số cặp (i, j) với i < j = {n * (n - 1) // 2:,}")

[2, 2, 1]    -> [1, 1, 0]  (số 2 không đếm số 2)
[-1, -1]     -> [0, 0]
Các cặp nghịch thế của [5, 2, 6, 1]: [(0, 1), (0, 3), (1, 3), (2, 3)]
Số nghịch thế = 4 = tổng counts = 4
n =      4: số cặp (i, j) với i < j = 6
n =   1000: số cặp (i, j) với i < j = 499,500
n = 100000: số cặp (i, j) với i < j = 4,999,950,000


## 3. Cách 1 — Brute force hai vòng lặp (trọng tài)

Cách người mới nghĩ ra đầu tiên, dịch thẳng từ đề:

- Vòng ngoài: chọn chỉ số `i` từ `0` tới `n − 1`.
- Vòng trong: duyệt mọi `j` từ `i + 1` tới `n − 1` (chỉ **bên phải** `i`).
- Nếu `nums[j] < nums[i]` (nhỏ hơn **hẳn**) thì `counts[i] += 1`.

`counts = [0] * n` tạo một list gồm `n` số 0 — "bảng đếm" ban đầu cho mọi vị trí.

Code này **chắc chắn đúng** vì làm y hệt định nghĩa. Ta giữ nó làm **"trọng tài"** để kiểm tra các cách nhanh hơn ở cuối notebook (test chéo trên hàng nghìn mảng ngẫu nhiên).

Độ phức tạp: vòng trong chạy `(n−1) + (n−2) + ... + 0 = n(n−1)/2` lần → **`O(n²)`** thời gian; bộ nhớ phụ **`O(1)`** (không tính mảng kết quả).

In [3]:
def count_smaller_brute(nums: List[int]) -> List[int]:
    n = len(nums)
    counts = [0] * n
    for i in range(n):
        for j in range(i + 1, n):        # chỉ nhìn sang BÊN PHẢI
            if nums[j] < nums[i]:        # nhỏ hơn HẲN (số bằng không tính)
                counts[i] += 1
    return counts

for nums, expected in examples:
    got = count_smaller_brute(nums)
    print(nums, "->", got, "| expected:", expected)
    assert got == expected

[5, 2, 6, 1] -> [2, 1, 1, 0] | expected: [2, 1, 1, 0]
[-1] -> [0] | expected: [0]
[-1, -1] -> [0, 0] | expected: [0, 0]


### Trace cách 1 trên Example 1

Mỗi dòng là một lần so sánh `nums[j] < nums[i]`:

| i | `nums[i]` | j | `nums[j]` | `nums[j] < nums[i]`? | `counts[i]` sau bước |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 0 | 5 | 1 | 2 | ✅ | 1 |
| 0 | 5 | 2 | 6 | ❌ | 1 |
| 0 | 5 | 3 | 1 | ✅ | **2** |
| 1 | 2 | 2 | 6 | ❌ | 0 |
| 1 | 2 | 3 | 1 | ✅ | **1** |
| 2 | 6 | 3 | 1 | ✅ | **1** |
| 3 | 1 | — | — | (không có j) | **0** |

Có đúng `4·3/2 = 6` lần so sánh. Để ý `i = 3`: vòng trong `range(4, 4)` rỗng nên `counts[3]` giữ nguyên `0` — phần tử cuối luôn có đáp án `0`.

In [4]:
def trace_brute(nums):
    n = len(nums)
    counts = [0] * n
    comparisons = 0
    for i in range(n):
        for j in range(i + 1, n):
            comparisons += 1
            ok = nums[j] < nums[i]
            if ok:
                counts[i] += 1
            print(f"i={i} (nums[i]={nums[i]}), j={j} (nums[j]={nums[j]}): "
                  f"{nums[j]} < {nums[i]}? {'✅' if ok else '❌'}  counts={counts}")
    print("Số lần so sánh:", comparisons)
    return counts

print("Kết quả:", trace_brute([5, 2, 6, 1]))

i=0 (nums[i]=5), j=1 (nums[j]=2): 2 < 5? ✅  counts=[1, 0, 0, 0]
i=0 (nums[i]=5), j=2 (nums[j]=6): 6 < 5? ❌  counts=[1, 0, 0, 0]
i=0 (nums[i]=5), j=3 (nums[j]=1): 1 < 5? ✅  counts=[2, 0, 0, 0]
i=1 (nums[i]=2), j=2 (nums[j]=6): 6 < 2? ❌  counts=[2, 0, 0, 0]
i=1 (nums[i]=2), j=3 (nums[j]=1): 1 < 2? ✅  counts=[2, 1, 0, 0]
i=2 (nums[i]=6), j=3 (nums[j]=1): 1 < 6? ✅  counts=[2, 1, 1, 0]
Số lần so sánh: 6
Kết quả: [2, 1, 1, 0]


### Cách 1 vướng ở đâu?

`O(n²)` với `n = 10^5` là khoảng `5·10^9` phép so sánh. Python thuần chạy cỡ vài chục triệu phép đơn giản mỗi giây → cần **hàng trăm giây**, trong khi LeetCode chỉ cho vài giây → **Time Limit Exceeded (TLE)**.

Cell dưới đo thời gian với `n = 2000` rồi ước lượng: `n` gấp 50 lần thì `n²` gấp `50² = 2500` lần (con số in ra phụ thuộc máy, nhưng chắc chắn quá chậm).

**Nút thắt:** với mỗi `i`, vòng trong **quét lại từ đầu** toàn bộ phần bên phải. Nhưng phần bên phải của `i` và của `i + 1` gần như **giống hệt nhau** (chỉ khác đúng một phần tử `nums[i+1]`)! Ta đang lãng phí công sức nhìn đi nhìn lại cùng một nhóm số. Mục tiếp theo khai thác đúng điểm này.

In [5]:
import random
import time

random.seed(42)
n = 2000
nums = [random.randint(-10**4, 10**4) for _ in range(n)]
start = time.perf_counter()
count_smaller_brute(nums)
elapsed = time.perf_counter() - start
print(f"n = {n}: mất khoảng {elapsed:.3f} giây")
print(f"Ước lượng cho n = 100000 (gấp 2500 lần): khoảng {elapsed * 2500:.0f} giây -> TLE")

n = 2000: mất khoảng 0.097 giây
Ước lượng cho n = 100000 (gấp 2500 lần): khoảng 244 giây -> TLE


## 4. Bước đột phá 1 — Duyệt từ phải sang trái với một "túi" số đã gặp

So sánh phần bên phải của các vị trí liên tiếp trong `[5, 2, 6, 1]`:

```text
bên phải của i=3:  []
bên phải của i=2:  [1]           = bên phải của i=3 + thêm nums[3]
bên phải của i=1:  [6, 1]        = bên phải của i=2 + thêm nums[2]
bên phải của i=0:  [2, 6, 1]     = bên phải của i=1 + thêm nums[1]
```

Nếu ta **duyệt `i` từ phải sang trái** (`i = n−1, n−2, ..., 0`), thì "phần bên phải của `i`" chính là **tập các số ta đã đi qua**. Ta giữ chúng trong một "**túi**" (một cấu trúc dữ liệu), và mỗi bước chỉ làm 2 việc:

1. **Hỏi túi:** "trong túi có bao nhiêu số `< nums[i]`?" → đó là `counts[i]`.
2. **Bỏ `nums[i]` vào túi** (để các vị trí bên trái dùng).

⚠️ **Thứ tự quan trọng: hỏi TRƯỚC, bỏ vào SAU.** Nếu bỏ `nums[i]` vào trước rồi mới hỏi thì túi chứa cả chính nó — với `<` thì vô hại, nhưng thói quen "hỏi trước, thêm sau" giúp bạn tránh lỗi ở các bài tương tự (vd. đếm `<=`).

`range(n - 1, -1, -1)` đọc là "bắt đầu từ `n−1`, dừng **trước** `-1` (tức dừng sau khi chạy `0`), mỗi bước `-1`" → sinh `n−1, n−2, ..., 0`.

Trạng thái túi trên `[5, 2, 6, 1]`:

| i | `nums[i]` | Túi trước khi hỏi | Số trong túi `< nums[i]` | `counts[i]` | Túi sau khi thêm |
|:---:|:---:|:---|:---|:---:|:---|
| 3 | 1 | `[]` | (không có) | 0 | `[1]` |
| 2 | 6 | `[1]` | 1 | 1 | `[1, 6]` |
| 1 | 2 | `[1, 6]` | 1 | 1 | `[1, 6, 2]` |
| 0 | 5 | `[1, 6, 2]` | 1, 2 | 2 | `[1, 6, 2, 5]` |

Nếu túi chỉ là một list bình thường (chưa sắp xếp) thì "hỏi túi" vẫn phải duyệt cả túi → vẫn `O(n²)`. Nhưng bây giờ bài toán đã được **tách thành một câu hỏi thiết kế rõ ràng**: *cần một cái túi hỗ trợ nhanh 2 thao tác `thêm(x)` và `đếm_nhỏ_hơn(x)`*. Cell dưới viết khung này với túi "ngây thơ".

In [6]:
def count_smaller_bag_naive(nums: List[int], verbose: bool = False) -> List[int]:
    n = len(nums)
    counts = [0] * n
    bag = []                                   # túi: các số bên phải vị trí hiện tại
    for i in range(n - 1, -1, -1):             # i = n-1, n-2, ..., 0
        x = nums[i]
        counts[i] = sum(1 for y in bag if y < x)   # 1) HỎI túi (duyệt cả túi: O(len(bag)))
        if verbose:
            print(f"i={i}, x={x}, túi={bag} -> counts[{i}] = {counts[i]}")
        bag.append(x)                          # 2) THÊM vào túi
    return counts

print("Kết quả:", count_smaller_bag_naive([5, 2, 6, 1], verbose=True))
for nums, expected in examples:
    assert count_smaller_bag_naive(nums) == expected
print("Khung 'duyệt từ phải + túi' đúng trên cả 3 example ✅")

i=3, x=1, túi=[] -> counts[3] = 0
i=2, x=6, túi=[1] -> counts[2] = 1
i=1, x=2, túi=[1, 6] -> counts[1] = 1
i=0, x=5, túi=[1, 6, 2] -> counts[0] = 2
Kết quả: [2, 1, 1, 0]
Khung 'duyệt từ phải + túi' đúng trên cả 3 example ✅


### Bản đồ các lời giải phía sau

Mọi cách nhanh đều là câu trả lời cho câu hỏi "túi nên là gì?" — hoặc đổi hẳn góc nhìn:

| Cách | Ý tưởng | `thêm(x)` | `đếm_nhỏ_hơn(x)` | Tổng |
|---|---|---|---|---|
| Túi ngây thơ (ở trên) | list thường | `O(1)` | `O(n)` | `O(n²)` |
| **2** — list sắp xếp + `bisect` | giữ túi luôn sắp xếp, tìm nhị phân | `O(n)` (dịch phần tử) | `O(log n)` | `O(n²)` worst, nhưng nhanh trong thực tế |
| **3** — merge sort (cách chính) | không dùng túi: đếm theo **nhóm** trong lúc trộn | — | — | `O(n log n)` |
| **4** — Fenwick Tree + nén tọa độ | túi = mảng đếm theo hạng, BIT tính tổng tiền tố | `O(log n)` | `O(log n)` | `O(n log n)` |
| **5** — BST tự cài (tùy chọn) | túi = cây tìm kiếm có đếm cây con trái | `O(độ cao)` | gộp cùng lúc thêm | `O(n log n)` trung bình, `O(n²)` worst |

Trước hết, ta cần một công cụ nền: **tìm kiếm nhị phân**.

---

## 5. Kiến thức nền 1 — Tìm kiếm nhị phân (binary search) và module `bisect`

Giả sử túi được giữ **sắp xếp tăng dần**, ví dụ `bag = [1, 2, 2, 6]`, và ta hỏi "có bao nhiêu số `< 5`?". Vì list đã sắp xếp, các số `< 5` luôn **dồn hết về đầu list**: `[1, 2, 2 | 6]`. Vậy câu trả lời = **vị trí của "vạch ngăn"** = vị trí mà ta sẽ chèn `5` vào để list vẫn sắp xếp = `3`.

Tìm vạch ngăn không cần duyệt từng phần tử: nhìn phần tử **ở giữa**; nếu nó `< x` thì vạch ngăn nằm bên phải nó (bỏ cả nửa trái), ngược lại vạch ngăn nằm ở nó hoặc bên trái (bỏ nửa phải). Mỗi bước loại **một nửa** → chỉ cần khoảng `log2(n)` bước (với `n = 10^5` là khoảng **17** bước thay vì 100 000). Đó là **binary search**.

Python có sẵn module `bisect` (thư viện chuẩn) với hai hàm, cho list `a` **đã sắp xếp**:

- `bisect_left(a, x)` = vị trí **đầu tiên** mà `a[pos] >= x` = **số phần tử `< x`** (nếu có số bằng `x`, vạch ngăn đứng **bên trái** chúng).
- `bisect_right(a, x)` = vị trí **đầu tiên** mà `a[pos] > x` = **số phần tử `<= x`** (vạch ngăn đứng **bên phải** các số bằng `x`).

Ví dụ `a = [1, 2, 2, 6]`:

| x | `bisect_left(a, x)` | `bisect_right(a, x)` | Giải thích |
|:---:|:---:|:---:|---|
| 5 | 3 | 3 | không có số 5, hai hàm giống nhau |
| 2 | **1** | **3** | có hai số 2: left đứng trước chúng, right đứng sau |
| 0 | 0 | 0 | mọi số đều `>= 0` |
| 9 | 4 | 4 | mọi số đều `< 9` |

**Bài này đếm "nhỏ hơn hẳn" nên dùng `bisect_left`.** Dùng nhầm `bisect_right` sẽ đếm luôn cả số bằng → sai đúng ở ca số trùng.

In [7]:
from bisect import bisect_left, bisect_right, insort

a = [1, 2, 2, 6]
for x in [5, 2, 0, 9]:
    print(f"x={x}: bisect_left={bisect_left(a, x)} (số phần tử < x), "
          f"bisect_right={bisect_right(a, x)} (số phần tử <= x)")

# Kiểm chứng bằng cách đếm trực tiếp
for x in [5, 2, 0, 9]:
    assert bisect_left(a, x) == sum(1 for y in a if y < x)
    assert bisect_right(a, x) == sum(1 for y in a if y <= x)
print("bisect_left = số phần tử < x;  bisect_right = số phần tử <= x  ✅")

x=5: bisect_left=3 (số phần tử < x), bisect_right=3 (số phần tử <= x)
x=2: bisect_left=1 (số phần tử < x), bisect_right=3 (số phần tử <= x)
x=0: bisect_left=0 (số phần tử < x), bisect_right=0 (số phần tử <= x)
x=9: bisect_left=4 (số phần tử < x), bisect_right=4 (số phần tử <= x)
bisect_left = số phần tử < x;  bisect_right = số phần tử <= x  ✅


### Tự cài `bisect_left` để hiểu nó hoạt động thế nào

Ta giữ hai biến `lo`, `hi` với **bất biến (invariant)**: *vạch ngăn (đáp án) luôn nằm trong đoạn `[lo, hi]`*. Ban đầu `lo = 0`, `hi = len(a)` (đáp án có thể là `len(a)` nếu mọi số đều `< x`).

Mỗi vòng, xét `mid = (lo + hi) // 2` (`//` là **chia lấy phần nguyên**: `7 // 2 = 3`):

- Nếu `a[mid] < x`: `a[mid]` còn nằm bên trái vạch ngăn → đáp án `> mid` → `lo = mid + 1`.
- Ngược lại (`a[mid] >= x`): `mid` có thể chính là vạch ngăn → đáp án `<= mid` → `hi = mid` (**không** `mid − 1`, kẻo bỏ mất đáp án).

Khi `lo == hi`, đoạn chỉ còn một vị trí → đó là đáp án. Mỗi vòng đoạn `[lo, hi]` co lại ít nhất một nửa nên vòng lặp chắc chắn dừng sau khoảng `log2(n)` vòng.

Ví dụ `a = [1, 2, 2, 5, 6, 8, 9]`, `x = 5`:

| Vòng | lo | hi | mid | `a[mid]` | So sánh | Cập nhật |
|:---:|:---:|:---:|:---:|:---:|---|---|
| 1 | 0 | 7 | 3 | 5 | `5 < 5`? Không | `hi = 3` |
| 2 | 0 | 3 | 1 | 2 | `2 < 5`? Có | `lo = 2` |
| 3 | 2 | 3 | 2 | 2 | `2 < 5`? Có | `lo = 3` |
| — | 3 | 3 | | | `lo == hi` → dừng | đáp án **3** |

Có đúng 3 số `< 5` là `1, 2, 2` ✅.

In [8]:
def my_bisect_left(a, x, verbose=False):
    lo, hi = 0, len(a)                   # invariant: đáp án nằm trong [lo, hi]
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        step += 1
        if a[mid] < x:
            action = f"a[{mid}]={a[mid]} < {x} -> lo = {mid + 1}"
            lo = mid + 1                 # đáp án nằm bên phải mid
        else:
            action = f"a[{mid}]={a[mid]} >= {x} -> hi = {mid}"
            hi = mid                     # mid có thể là đáp án, bỏ nửa phải
        if verbose:
            print(f"  vòng {step}: {action}  => [lo, hi] = [{lo}, {hi}]")
    return lo

print("my_bisect_left([1, 2, 2, 5, 6, 8, 9], 5):")
print("  đáp án =", my_bisect_left([1, 2, 2, 5, 6, 8, 9], 5, verbose=True))

random.seed(1)
for _ in range(2000):
    a = sorted(random.randint(-5, 5) for _ in range(random.randint(0, 8)))
    x = random.randint(-6, 6)
    assert my_bisect_left(a, x) == bisect_left(a, x)
print("my_bisect_left khớp bisect.bisect_left trên 2000 test ngẫu nhiên ✅")

my_bisect_left([1, 2, 2, 5, 6, 8, 9], 5):
  vòng 1: a[3]=5 >= 5 -> hi = 3  => [lo, hi] = [0, 3]
  vòng 2: a[1]=2 < 5 -> lo = 2  => [lo, hi] = [2, 3]
  vòng 3: a[2]=2 < 5 -> lo = 3  => [lo, hi] = [3, 3]
  đáp án = 3
my_bisect_left khớp bisect.bisect_left trên 2000 test ngẫu nhiên ✅


### `insort`: chèn mà vẫn giữ sắp xếp — và cái giá ẩn của nó

`insort(a, x)` (cũng trong `bisect`) chèn `x` vào list `a` sao cho `a` vẫn sắp xếp. Nó làm 2 việc:

1. **Tìm vị trí** bằng binary search — `O(log n)`.
2. **Chèn** vào vị trí đó bằng `a.insert(pos, x)`.

Bước 2 mới là cái giá ẩn: list của Python là một **mảng liên tiếp trong bộ nhớ** (các ô nằm sát nhau). Muốn chèn vào giữa, mọi phần tử **phía sau** vị trí chèn phải **dịch sang phải một ô** để lấy chỗ → tốn `O(n)` trong trường hợp xấu (chèn vào đầu list). Ví dụ chèn `0` vào `[1, 2, 2, 6]`: cả bốn số `1, 2, 2, 6` đều phải dịch.

Tuy vậy, việc dịch này được CPython làm bằng một lệnh sao chép bộ nhớ khối (`memmove`) viết bằng C, nhanh hơn vòng lặp Python **rất nhiều lần**. Vì vậy trong thực tế `insort` thường "nhanh như `O(log n)`" với `n` cỡ `10^5` — nhưng về lý thuyết vẫn là `O(n)` mỗi lần.

In [9]:
bag = []
for x in [1, 6, 2, 5]:                  # đúng thứ tự thêm vào túi ở mục 4
    pos = bisect_left(bag, x)
    insort(bag, x)
    print(f"insort {x}: chèn ở vị trí {pos}, dịch {len(bag) - 1 - pos} phần tử phía sau -> {bag}")

big = list(range(1, 10**5 + 1))         # list 100 000 phần tử đã sắp xếp
t = time.perf_counter(); big.insert(0, 0); front = time.perf_counter() - t
t = time.perf_counter(); big.append(10**6); back = time.perf_counter() - t
print(f"Chèn vào ĐẦU list 100 000 phần tử: {front * 1e6:.1f} µs (dịch 100 000 ô)")
print(f"Thêm vào CUỐI list:               {back * 1e6:.1f} µs (không phải dịch)")

insort 1: chèn ở vị trí 0, dịch 0 phần tử phía sau -> [1]
insort 6: chèn ở vị trí 1, dịch 0 phần tử phía sau -> [1, 6]
insort 2: chèn ở vị trí 1, dịch 1 phần tử phía sau -> [1, 2, 6]
insort 5: chèn ở vị trí 2, dịch 1 phần tử phía sau -> [1, 2, 5, 6]
Chèn vào ĐẦU list 100 000 phần tử: 253.2 µs (dịch 100 000 ô)
Thêm vào CUỐI list:               38.3 µs (không phải dịch)


## 6. Cách 2 — Duyệt từ phải, giữ list `sorted_seen` + `bisect`

Ghép khung "túi" ở mục 4 với `bisect`:

```python
sorted_seen = []                          # túi, luôn sắp xếp tăng dần
for i in range(n - 1, -1, -1):
    counts[i] = bisect_left(sorted_seen, nums[i])   # 1) số phần tử < nums[i]   — O(log n)
    insort(sorted_seen, nums[i])                    # 2) thêm vào, giữ sắp xếp  — O(n) dịch phần tử
```

**Số âm** không gây vấn đề gì: `bisect` chỉ so sánh `<`, không dùng giá trị làm chỉ số. **Số trùng** được xử lý đúng nhờ `bisect_left` (đếm `<` chứ không đếm `<=`).

Trace trên `nums = [5, 2, 6, 1, 2]` (có hai số `2` để thấy nhánh số trùng). Đáp án đúng (tính tay bằng định nghĩa): `[3, 1, 2, 0, 0]`.

| i | `nums[i]` | `sorted_seen` trước | `bisect_left` | `counts[i]` | `sorted_seen` sau |
|:---:|:---:|:---|:---:|:---:|:---|
| 4 | 2 | `[]` | 0 | 0 | `[2]` |
| 3 | 1 | `[2]` | 0 | 0 | `[1, 2]` |
| 2 | 6 | `[1, 2]` | 2 | 2 | `[1, 2, 6]` |
| 1 | 2 | `[1, 2, 6]` | **1** | **1** | `[1, 2, 2, 6]` |
| 0 | 5 | `[1, 2, 2, 6]` | 3 | 3 | `[1, 2, 2, 5, 6]` |

Dòng `i = 1` là dòng đáng chú ý: túi đã có một số `2`; `bisect_left` đặt vạch ngăn **trước** số `2` đó nên chỉ đếm số `1` → `counts[1] = 1`. Nếu dùng `bisect_right` sẽ ra `2` — sai.

In [10]:
def count_smaller_bisect(nums: List[int]) -> List[int]:
    n = len(nums)
    counts = [0] * n
    sorted_seen = []                                # các số bên phải, luôn sắp xếp tăng dần
    for i in range(n - 1, -1, -1):
        counts[i] = bisect_left(sorted_seen, nums[i])   # số phần tử < nums[i]
        insort(sorted_seen, nums[i])                    # thêm nums[i] mà vẫn giữ sắp xếp
    return counts

for nums, expected in examples:
    got = count_smaller_bisect(nums)
    print(nums, "->", got, "| expected:", expected)
    assert got == expected
print("[5, 2, 6, 1, 2] ->", count_smaller_bisect([5, 2, 6, 1, 2]))

[5, 2, 6, 1] -> [2, 1, 1, 0] | expected: [2, 1, 1, 0]
[-1] -> [0] | expected: [0]
[-1, -1] -> [0, 0] | expected: [0, 0]
[5, 2, 6, 1, 2] -> [3, 1, 2, 0, 0]


In [11]:
def trace_bisect(nums):
    n = len(nums)
    counts = [0] * n
    sorted_seen = []
    for i in range(n - 1, -1, -1):
        before = sorted_seen[:]                     # chụp lại trạng thái trước khi thêm
        pos = bisect_left(sorted_seen, nums[i])
        counts[i] = pos
        insort(sorted_seen, nums[i])
        print(f"i={i}, nums[i]={nums[i]}: seen trước={before}, bisect_left={pos}, "
              f"(bisect_right sẽ là {bisect_right(before, nums[i])}) -> seen sau={sorted_seen}")
    return counts

print("Kết quả:", trace_bisect([5, 2, 6, 1, 2]))

i=4, nums[i]=2: seen trước=[], bisect_left=0, (bisect_right sẽ là 0) -> seen sau=[2]
i=3, nums[i]=1: seen trước=[2], bisect_left=0, (bisect_right sẽ là 0) -> seen sau=[1, 2]
i=2, nums[i]=6: seen trước=[1, 2], bisect_left=2, (bisect_right sẽ là 2) -> seen sau=[1, 2, 6]
i=1, nums[i]=2: seen trước=[1, 2, 6], bisect_left=1, (bisect_right sẽ là 2) -> seen sau=[1, 2, 2, 6]
i=0, nums[i]=5: seen trước=[1, 2, 2, 6], bisect_left=3, (bisect_right sẽ là 3) -> seen sau=[1, 2, 2, 5, 6]
Kết quả: [3, 1, 2, 0, 0]


### Độ phức tạp của Cách 2 — đừng nhầm là `O(n log n)`!

Gọi `n = len(nums)`.

- `bisect_left`: `O(log n)` mỗi lần.
- `insort`: tìm vị trí `O(log n)` **+ dịch phần tử `O(n)`** mỗi lần.
- Tổng: **`O(n²)` trong trường hợp xấu nhất** (về lý thuyết), bộ nhớ `O(n)`.

Trường hợp xấu nhất là khi **mỗi số mới đều phải chèn vào đầu túi**, tức là duyệt từ phải sang trái gặp các số **giảm dần** — nghĩa là `nums` **tăng dần**, ví dụ `[1, 2, 3, ..., n]`. Khi đó lần chèn thứ `k` phải dịch `k` phần tử → tổng `≈ n²/2` lần dịch.

Trên thực tế, nhờ `memmove` viết bằng C, Cách 2 với `n = 10^5` chỉ mất cỡ dưới 1–2 giây và thường **được LeetCode chấp nhận**. Nhưng khi phỏng vấn, hãy nói rõ: *"`insort` trên list là `O(n)` mỗi lần; muốn `O(log n)` thật sự cho cả thêm lẫn đếm thì cần một **cây tìm kiếm cân bằng** (balanced BST: AVL, red-black, treap...) hoặc cấu trúc như `SortedList` của thư viện `sortedcontainers` (không thuộc thư viện chuẩn), hoặc Fenwick Tree ở Cách 4."*

Cell dưới đo thử trên dữ liệu ngẫu nhiên và trên trường hợp xấu nhất (con số phụ thuộc máy).

In [12]:
random.seed(7)
n = 10**5
random_nums = [random.randint(-10**4, 10**4) for _ in range(n)]
worst_nums = list(range(n))             # tăng dần: duyệt từ phải sẽ luôn chèn vào ĐẦU túi

for name, data in [("ngẫu nhiên", random_nums), ("tăng dần (xấu nhất)", worst_nums)]:
    t = time.perf_counter()
    count_smaller_bisect(data)
    print(f"Cách 2, n = {n:,}, dữ liệu {name:<20}: {time.perf_counter() - t:.2f} giây")
print("-> Nhanh trong thực tế, nhưng về lý thuyết vẫn O(n²) vì mỗi insort dịch tới O(n) phần tử.")

Cách 2, n = 100,000, dữ liệu ngẫu nhiên          : 0.60 giây


Cách 2, n = 100,000, dữ liệu tăng dần (xấu nhất) : 0.80 giây
-> Nhanh trong thực tế, nhưng về lý thuyết vẫn O(n²) vì mỗi insort dịch tới O(n) phần tử.


## 7. Kiến thức nền 2 — Merge sort (sắp xếp trộn) từ căn bản

Cách 2 hỏi **từng phần tử một** "bên phải có bao nhiêu số nhỏ hơn mày?". Cách 3 đổi góc nhìn: đếm **theo nhóm** trong lúc **sắp xếp** mảng. Muốn hiểu, trước hết phải hiểu **merge sort**.

Merge sort dựa trên chiến lược **chia để trị (divide and conquer)**:

1. **Chia:** cắt mảng làm đôi tại giữa.
2. **Trị:** sắp xếp **mỗi nửa** (bằng chính merge sort — gọi lại hàm, gọi là **đệ quy**).
3. **Trộn (merge):** gộp hai nửa **đã sắp xếp** thành một mảng sắp xếp.

Ta học **bước 3** trước vì nó là trái tim của cả thuật toán (và của mẹo đếm).

### Trộn hai list đã sắp xếp bằng hai con trỏ (two pointers)

Cho `left = [2, 5]` và `right = [1, 6]`, cả hai đã tăng dần. Muốn có `[1, 2, 5, 6]`:

- Con trỏ `i` chỉ vào phần tử nhỏ nhất **chưa lấy** của `left`, `j` chỉ vào phần tử nhỏ nhất chưa lấy của `right`.
- Mỗi bước, so sánh `left[i]` với `right[j]`, **lấy số nhỏ hơn** đưa xuống kết quả và tiến con trỏ đó.
- Khi một bên hết, chép nốt phần còn lại của bên kia.

| Bước | `left[i]` | `right[j]` | Lấy | `merged` |
|:---:|:---:|:---:|:---:|:---|
| 1 | 2 | 1 | 1 (từ right) | `[1]` |
| 2 | 2 | 6 | 2 (từ left) | `[1, 2]` |
| 3 | 5 | 6 | 5 (từ left) | `[1, 2, 5]` |
| 4 | (hết) | 6 | 6 (chép nốt) | `[1, 2, 5, 6]` |

Vì sao đúng? Số nhỏ nhất còn lại của cả hai list **chắc chắn** là `left[i]` hoặc `right[j]` (mỗi list đã tăng dần, nên số đầu là số nhỏ nhất của nó). Mỗi phần tử được lấy đúng một lần → trộn mất `O(len(left) + len(right))`.

In [13]:
def merge_two_sorted(left, right, verbose=False):
    merged = []
    i, j = 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:              # bằng nhau thì ưu tiên lấy bên trái
            merged.append(left[i]); src = "left"; i += 1
        else:
            merged.append(right[j]); src = "right"; j += 1
        if verbose:
            print(f"  lấy {merged[-1]} từ {src:<5} -> merged = {merged}")
    merged.extend(left[i:])                  # chép nốt phần còn lại (chỉ một trong hai còn)
    merged.extend(right[j:])
    if verbose:
        print(f"  chép nốt phần còn lại      -> merged = {merged}")
    return merged

print("Trộn [2, 5] và [1, 6]:")
merge_two_sorted([2, 5], [1, 6], verbose=True)
print("Trộn [-3, 0, 0, 7] và [-1, 0, 9]:", merge_two_sorted([-3, 0, 0, 7], [-1, 0, 9]))

Trộn [2, 5] và [1, 6]:
  lấy 1 từ right -> merged = [1]
  lấy 2 từ left  -> merged = [1, 2]
  lấy 5 từ left  -> merged = [1, 2, 5]
  chép nốt phần còn lại      -> merged = [1, 2, 5, 6]
Trộn [-3, 0, 0, 7] và [-1, 0, 9]: [-3, -1, 0, 0, 0, 7, 9]


### Đệ quy (recursion) và merge sort đầy đủ

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Mọi hàm đệ quy cần:

- **Base case (trường hợp cơ sở):** bài toán đủ nhỏ để trả lời ngay, không gọi tiếp. Ở merge sort: mảng có `0` hoặc `1` phần tử thì **đã sắp xếp sẵn**.
- **Bước đệ quy:** chia thành bài toán nhỏ hơn, gọi lại hàm, rồi ghép kết quả.

Cây đệ quy của `merge_sort([5, 2, 6, 1])`:

```text
                [5, 2, 6, 1]
               /            \
          [5, 2]            [6, 1]
          /    \            /    \
        [5]    [2]        [6]    [1]        <- base case
          \    /            \    /
          [2, 5]            [1, 6]          <- trộn
               \            /
                [1, 2, 5, 6]                <- trộn
```

Cell dưới in thụt lề theo độ sâu đệ quy để bạn nhìn thấy thứ tự các lời gọi.

In [14]:
def merge_sort(a, depth=0, verbose=False):
    pad = "    " * depth
    if len(a) <= 1:                                  # base case
        if verbose:
            print(f"{pad}{a} -> base case")
        return a[:]
    mid = len(a) // 2
    if verbose:
        print(f"{pad}{a}: chia thành {a[:mid]} và {a[mid:]}")
    left = merge_sort(a[:mid], depth + 1, verbose)   # sắp xếp nửa trái
    right = merge_sort(a[mid:], depth + 1, verbose)  # sắp xếp nửa phải
    merged = merge_two_sorted(left, right)           # trộn
    if verbose:
        print(f"{pad}trộn {left} + {right} -> {merged}")
    return merged

merge_sort([5, 2, 6, 1], verbose=True)

random.seed(3)
for _ in range(500):
    a = [random.randint(-9, 9) for _ in range(random.randint(0, 12))]
    assert merge_sort(a) == sorted(a)
print("merge_sort khớp sorted() trên 500 test ngẫu nhiên ✅")

[5, 2, 6, 1]: chia thành [5, 2] và [6, 1]
    [5, 2]: chia thành [5] và [2]
        [5] -> base case
        [2] -> base case
    trộn [5] + [2] -> [2, 5]
    [6, 1]: chia thành [6] và [1]
        [6] -> base case
        [1] -> base case
    trộn [6] + [1] -> [1, 6]
trộn [2, 5] + [1, 6] -> [1, 2, 5, 6]
merge_sort khớp sorted() trên 500 test ngẫu nhiên ✅


### Độ phức tạp của merge sort: `O(n log n)`

Nhìn cây đệ quy theo **từng tầng**:

- Tầng trên cùng: 1 lần trộn tổng cộng `n` phần tử.
- Tầng 2: 2 lần trộn, mỗi lần `n/2` phần tử → vẫn tổng `n`.
- Tầng 3: 4 lần trộn, mỗi lần `n/4` → tổng `n`.
- ...

Mỗi tầng tốn `O(n)`; mỗi lần đi xuống, kích thước giảm một nửa nên có khoảng `log2(n)` tầng (với `n = 10^5` là khoảng 17 tầng). Tổng: **`O(n log n)`** — với `n = 10^5` là cỡ `1.7·10^6` thao tác, so với `5·10^9` của brute force.

Bộ nhớ: `O(n)` cho các list tạm khi trộn, cộng `O(log n)` cho ngăn xếp đệ quy (độ sâu ≈ 17 — không sợ vượt giới hạn đệ quy của Python, mặc định 1000).

---

## 8. Bước đột phá 2 — Đếm trong lúc trộn: các phần tử bên phải "vượt lên"

Xét một bước trộn bất kỳ: mảng con được chia thành **nửa trái** `L` và **nửa phải** `R` (theo **vị trí gốc**: mọi phần tử của `R` đều đứng **bên phải** mọi phần tử của `L` trong mảng ban đầu).

Lấy một phần tử `x` thuộc `L`. Các số nhỏ hơn `x` nằm bên phải `x` chia làm hai nhóm:

- **Nhóm (a): nằm cùng nửa trái `L`, bên phải `x`** → đã được đếm khi đệ quy xử lý `L` (ở tầng sâu hơn).
- **Nhóm (b): nằm ở nửa phải `R`** → **mọi** phần tử của `R` đều ở bên phải `x`, nên chỉ cần đếm **số phần tử của `R` nhỏ hơn `x`**.

Đếm nhóm (b) ngay khi trộn: hai nửa đã được sắp xếp; mỗi khi một phần tử của `R` được đưa xuống kết quả **trước** `x`, nghĩa là nó **nhỏ hơn `x`** — nó đã "**vượt lên**" trước `x`. Vậy:

> **Khi đưa `x` (từ nửa trái) xuống kết quả, số phần tử nửa phải đã được đưa xuống trước đó chính là số phần tử của `R` nhỏ hơn `x`. Cộng con số đó vào `counts` của `x`.**

Với con trỏ `j` chạy trên nửa phải bắt đầu từ `mid`, con số đó là `j − mid`.

**Số bằng nhau thì sao?** Nếu `x` (trái) bằng `y` (phải), ta phải **lấy `x` trước** (điều kiện `<=` khi ưu tiên bên trái) để `y` **không** được tính là "đã vượt lên" — vì `y` không nhỏ hơn hẳn `x`. Đây chính là quy tắc "bằng nhau ưu tiên bên trái" của `merge_two_sorted` ở trên.

**Phần tử của nửa phải thì sao?** Khi đưa `y ∈ R` xuống, ta **không cộng gì** cho nó: các phần tử của `L` đều nằm **bên trái** `y`, không liên quan tới `counts` của `y`. Phần bên phải của `y` (nằm trong `R`) đã được xử lý ở tầng đệ quy sâu hơn.

Mỗi cặp `(i, j)` với `i < j` "gặp nhau" ở **đúng một** bước trộn: bước trộn nhỏ nhất mà `i` ở nửa trái và `j` ở nửa phải. Vì vậy mỗi cặp nghịch thế được đếm **đúng một lần** — không thiếu, không trùng.

### Vì sao phải sắp xếp **mảng chỉ số** chứ không sắp xếp `nums`?

Khi sắp xếp, các phần tử bị **dời chỗ**. Nếu ta sắp xếp chính `nums`, đến lúc muốn cộng vào `counts[...]` ta không còn biết số này **ban đầu nằm ở vị trí nào**.

Ý tưởng "dùng dict tra ngược `giá trị → vị trí`" **thất bại khi có số trùng**: với `[2, -1, 2, -1]`, giá trị `2` ứng với cả vị trí `0` lẫn `2`.

Giải pháp: tạo **mảng chỉ số** `idx = [0, 1, 2, ..., n−1]` và sắp xếp **`idx`** theo giá trị `nums[idx[k]]`. Mỗi phần tử của `idx` "mang theo" vị trí gốc của nó, nên lúc nào ta cũng biết cộng vào `counts[idx[k]]`. Số trùng không còn là vấn đề vì mỗi vị trí là một chỉ số riêng.

Cú pháp hữu ích: `sorted(range(n), key=lambda k: nums[k])` sắp xếp các chỉ số theo giá trị mà chúng trỏ tới (`lambda k: nums[k]` là một **hàm vô danh** nhận `k` và trả về `nums[k]`).

In [15]:
nums = [2, -1, 2, -1]
value_to_index = {v: k for k, v in enumerate(nums)}       # dict giá trị -> vị trí
print("dict giá trị -> vị trí:", value_to_index, " <- mất vị trí 0 và 1 vì giá trị trùng!")

idx = list(range(len(nums)))
idx_sorted = sorted(idx, key=lambda k: nums[k])           # sắp xếp CHỈ SỐ theo giá trị
print("idx ban đầu     :", idx)
print("idx sau sắp xếp :", idx_sorted)
print("giá trị tương ứng:", [nums[k] for k in idx_sorted], " <- tăng dần, mỗi chỉ số vẫn nhớ vị trí gốc")

dict giá trị -> vị trí: {2: 2, -1: 3}  <- mất vị trí 0 và 1 vì giá trị trùng!
idx ban đầu     : [0, 1, 2, 3]
idx sau sắp xếp : [1, 3, 0, 2]
giá trị tương ứng: [-1, -1, 2, 2]  <- tăng dần, mỗi chỉ số vẫn nhớ vị trí gốc


### Thử tay một bước trộn có đếm

Lấy bước trộn cuối cùng của `[5, 2, 6, 1]`. Sau khi đệ quy xong hai nửa:

- Nửa trái (vị trí gốc 0, 1) đã sắp xếp: chỉ số `[1, 0]` ↔ giá trị `[2, 5]`. Lúc này `counts = [1, 0, ?, ?]` (số `5` đã đếm được số `2` ở tầng dưới).
- Nửa phải (vị trí gốc 2, 3) đã sắp xếp: chỉ số `[3, 2]` ↔ giá trị `[1, 6]`. Lúc này `counts[2] = 1` (số `6` đã đếm được số `1`).

Trộn, với `jumped` = số phần tử nửa phải đã "vượt lên":

| Bước | Trái đang xét | Phải đang xét | So sánh | Lấy | `jumped` | Cộng vào `counts` |
|:---:|:---:|:---:|---|:---:|:---:|---|
| 1 | 2 (idx 1) | 1 (idx 3) | `1 < 2` → phải vượt lên | 1 | **1** | — |
| 2 | 2 (idx 1) | 6 (idx 2) | `2 <= 6` → lấy trái | 2 | 1 | `counts[1] += 1` → 1 |
| 3 | 5 (idx 0) | 6 (idx 2) | `5 <= 6` → lấy trái | 5 | 1 | `counts[0] += 1` → 2 |
| 4 | (hết) | 6 (idx 2) | chép nốt bên phải | 6 | 1 | — |

Kết quả: `counts = [2, 1, 1, 0]` ✅. Số `5` nhận `+1` (số `1` vượt lên) ở bước này, cộng với `+1` (số `2`) từ tầng dưới = `2`.

In [16]:
def merge_with_count_demo(nums, left_idx, right_idx, counts):
    """Trộn hai danh sách CHỈ SỐ (đã sắp xếp theo giá trị), đếm phần tử bên phải vượt lên."""
    merged = []
    i, j = 0, 0
    while i < len(left_idx) or j < len(right_idx):
        take_left = j == len(right_idx) or (i < len(left_idx) and nums[left_idx[i]] <= nums[right_idx[j]])
        if take_left:
            k = left_idx[i]; i += 1
            counts[k] += j                    # j = số phần tử bên phải đã vượt lên
            print(f"lấy TRÁI  {nums[k]} (idx {k}): jumped = {j} -> counts[{k}] += {j} -> counts = {counts}")
        else:
            k = right_idx[j]; j += 1
            if i == len(left_idx):
                print(f"lấy PHẢI  {nums[k]} (idx {k}): bên trái đã hết, chép nốt")
            else:
                print(f"lấy PHẢI  {nums[k]} (idx {k}): nhỏ hơn hẳn -> vượt lên, jumped = {j}")
        merged.append(k)
    return merged

nums = [5, 2, 6, 1]
counts = [1, 0, 1, 0]                         # trạng thái sau khi đệ quy xong hai nửa
merged = merge_with_count_demo(nums, [1, 0], [3, 2], counts)
print("idx sau trộn:", merged, "| giá trị:", [nums[k] for k in merged], "| counts:", counts)

lấy PHẢI  1 (idx 3): nhỏ hơn hẳn -> vượt lên, jumped = 1
lấy TRÁI  2 (idx 1): jumped = 1 -> counts[1] += 1 -> counts = [1, 1, 1, 0]
lấy TRÁI  5 (idx 0): jumped = 1 -> counts[0] += 1 -> counts = [2, 1, 1, 0]
lấy PHẢI  6 (idx 2): bên trái đã hết, chép nốt
idx sau trộn: [3, 1, 0, 2] | giá trị: [1, 2, 5, 6] | counts: [2, 1, 1, 0]


## 9. Cách 3 — Merge sort trên mảng chỉ số (cách chính, `O(n log n)`)

Ghép mọi thứ lại. Hàm đệ quy `sort(lo, hi)` sắp xếp đoạn `idx[lo:hi]` (từ `lo` tới **trước** `hi`) theo giá trị và đồng thời cộng dồn vào `counts`:

1. **Base case:** `hi − lo <= 1` (0 hoặc 1 phần tử) → không làm gì.
2. `mid = (lo + hi) // 2`; gọi `sort(lo, mid)` và `sort(mid, hi)`.
3. **Trộn** `idx[lo:mid]` và `idx[mid:hi]` với hai con trỏ `i` (từ `lo`) và `j` (từ `mid`):
   - Nếu `nums[idx[j]] < nums[idx[i]]` (phải **nhỏ hơn hẳn**): lấy bên phải, `j += 1` — một phần tử vượt lên.
   - Ngược lại (trái `<=` phải): lấy bên trái và **`counts[idx[i]] += j − mid`**.
   - Khi bên phải hết mà bên trái còn: mỗi phần tử trái còn lại cũng được cộng `j − mid` (lúc này `j = hi`: **cả nửa phải** đã vượt lên).
   - Khi bên trái hết mà bên phải còn: chép nốt, không cộng gì.
4. Ghi kết quả trộn đè lên `idx[lo:hi]` (gán slicing `idx[lo:hi] = merged`).

⚠️ Đừng quên bước "bên phải hết mà bên trái còn" — đây là chỗ người mới hay quên cộng, khiến các số **lớn nhất** của nửa trái bị thiếu đáp án.

**Hàm lồng nhau (nested function):** `sort` được định nghĩa **bên trong** `count_smaller_merge_sort` nên nó "nhìn thấy" `nums`, `idx`, `counts` của hàm ngoài mà không cần truyền tham số. Ta chỉ **sửa phần tử** của các list đó (`counts[k] += ...`, `idx[lo:hi] = ...`), không gán lại tên biến, nên không cần từ khóa `nonlocal`.

In [17]:
def count_smaller_merge_sort(nums: List[int]) -> List[int]:
    n = len(nums)
    counts = [0] * n
    idx = list(range(n))                      # sắp xếp CHỈ SỐ, không sắp xếp nums

    def sort(lo, hi):                         # sắp xếp idx[lo:hi] theo giá trị, cộng dồn counts
        if hi - lo <= 1:                      # base case: 0 hoặc 1 phần tử
            return
        mid = (lo + hi) // 2
        sort(lo, mid)
        sort(mid, hi)
        merged = []
        i, j = lo, mid
        while i < mid and j < hi:
            if nums[idx[j]] < nums[idx[i]]:   # phải NHỎ HƠN HẲN -> vượt lên
                merged.append(idx[j])
                j += 1
            else:                             # trái <= phải -> đưa trái xuống, cộng số đã vượt lên
                counts[idx[i]] += j - mid
                merged.append(idx[i])
                i += 1
        while i < mid:                        # phải đã hết: cả nửa phải đều nhỏ hơn
            counts[idx[i]] += j - mid
            merged.append(idx[i])
            i += 1
        while j < hi:                         # trái đã hết: chép nốt, không cộng gì
            merged.append(idx[j])
            j += 1
        idx[lo:hi] = merged

    sort(0, n)
    return counts

for nums, expected in examples:
    got = count_smaller_merge_sort(nums)
    print(nums, "->", got, "| expected:", expected)
    assert got == expected
print("[5, 2, 6, 1, 2] ->", count_smaller_merge_sort([5, 2, 6, 1, 2]))

[5, 2, 6, 1] -> [2, 1, 1, 0] | expected: [2, 1, 1, 0]
[-1] -> [0] | expected: [0]
[-1, -1] -> [0, 0] | expected: [0, 0]
[5, 2, 6, 1, 2] -> [3, 1, 2, 0, 0]


### Trace đầy đủ Cách 3 trên `[5, 2, 6, 1]`

Thứ tự thực hiện (đệ quy xong nửa trái rồi mới tới nửa phải, cuối cùng trộn):

| # | Lời gọi | Trộn (chỉ số : giá trị) | Diễn biến | `counts` sau bước |
|:---:|---|---|---|---|
| 1 | `sort(0,2)` | `[0]:5` + `[1]:2` | `2 < 5` → 2 vượt lên (jumped=1); phải hết → `counts[0] += 1` | `[1, 0, 0, 0]` |
| 2 | `sort(2,4)` | `[2]:6` + `[3]:1` | `1 < 6` → 1 vượt lên (jumped=1); phải hết → `counts[2] += 1` | `[1, 0, 1, 0]` |
| 3 | `sort(0,4)` | `[1,0]:[2,5]` + `[3,2]:[1,6]` | 1 vượt lên; lấy 2 → `counts[1] += 1`; lấy 5 → `counts[0] += 1`; chép 6 | `[2, 1, 1, 0]` |

Sau cùng `idx = [3, 1, 0, 2]` (giá trị `[1, 2, 5, 6]` — đã sắp xếp), `counts = [2, 1, 1, 0]` ✅.

Hàm `trace_merge_sort` dưới in đúng các bước này, thụt lề theo độ sâu đệ quy.

In [18]:
def trace_merge_sort(nums):
    n = len(nums)
    counts = [0] * n
    idx = list(range(n))

    def show(ids):
        return "[" + ", ".join(f"{k}:{nums[k]}" for k in ids) + "]"

    def sort(lo, hi, depth):
        pad = "  " * depth
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        sort(lo, mid, depth + 1)
        sort(mid, hi, depth + 1)
        print(f"{pad}sort({lo},{hi}): trộn {show(idx[lo:mid])} + {show(idx[mid:hi])}   (chỉ số:giá trị)")
        merged, i, j = [], lo, mid
        while i < mid or j < hi:
            if i < mid and (j == hi or nums[idx[i]] <= nums[idx[j]]):
                counts[idx[i]] += j - mid
                print(f"{pad}  lấy trái {nums[idx[i]]} (idx {idx[i]}), jumped={j - mid} -> counts[{idx[i]}]={counts[idx[i]]}")
                merged.append(idx[i]); i += 1
            else:
                merged.append(idx[j]); j += 1
                print(f"{pad}  lấy phải {nums[merged[-1]]} (idx {merged[-1]}) -> jumped={j - mid}")
        idx[lo:hi] = merged
        print(f"{pad}  => idx[{lo}:{hi}] = {show(merged)}, counts = {counts}")

    sort(0, n, 0)
    return counts

print("Kết quả:", trace_merge_sort([5, 2, 6, 1]))

  sort(0,2): trộn [0:5] + [1:2]   (chỉ số:giá trị)
    lấy phải 2 (idx 1) -> jumped=1
    lấy trái 5 (idx 0), jumped=1 -> counts[0]=1
    => idx[0:2] = [1:2, 0:5], counts = [1, 0, 0, 0]
  sort(2,4): trộn [2:6] + [3:1]   (chỉ số:giá trị)
    lấy phải 1 (idx 3) -> jumped=1
    lấy trái 6 (idx 2), jumped=1 -> counts[2]=1
    => idx[2:4] = [3:1, 2:6], counts = [1, 0, 1, 0]
sort(0,4): trộn [1:2, 0:5] + [3:1, 2:6]   (chỉ số:giá trị)
  lấy phải 1 (idx 3) -> jumped=1
  lấy trái 2 (idx 1), jumped=1 -> counts[1]=1
  lấy trái 5 (idx 0), jumped=1 -> counts[0]=2
  lấy phải 6 (idx 2) -> jumped=2
  => idx[0:4] = [3:1, 1:2, 0:5, 2:6], counts = [2, 1, 1, 0]
Kết quả: [2, 1, 1, 0]


### Trace với số trùng và số âm: `[2, -1, 2, -1]`

Đáp án đúng (tính tay): số `2` đầu có bên phải `[-1, 2, -1]` → hai số `-1` nhỏ hơn → `2`; số `-1` thứ hai có bên phải `[2, -1]` → không có số nào **nhỏ hơn hẳn** → `0`; số `2` thứ ba có bên phải `[-1]` → `1`; số cuối → `0`. Vậy **`[2, 0, 1, 0]`**.

Chú ý bước trộn cuối cùng: hai nửa đã sắp xếp là `[1:-1, 0:2]` và `[3:-1, 2:2]`. Khi so `-1` (trái, idx 1) với `-1` (phải, idx 3): **bằng nhau → lấy trái trước** nên `-1` bên phải không bị tính là vượt lên, `counts[1]` giữ `0`. Tương tự `2` (idx 0) với `2` (idx 2). Số âm không ảnh hưởng gì vì merge sort chỉ **so sánh** giá trị, không dùng giá trị làm chỉ số.

In [19]:
print("Kết quả:", trace_merge_sort([2, -1, 2, -1]))
assert count_smaller_merge_sort([2, -1, 2, -1]) == count_smaller_brute([2, -1, 2, -1]) == [2, 0, 1, 0]

  sort(0,2): trộn [0:2] + [1:-1]   (chỉ số:giá trị)
    lấy phải -1 (idx 1) -> jumped=1
    lấy trái 2 (idx 0), jumped=1 -> counts[0]=1
    => idx[0:2] = [1:-1, 0:2], counts = [1, 0, 0, 0]
  sort(2,4): trộn [2:2] + [3:-1]   (chỉ số:giá trị)
    lấy phải -1 (idx 3) -> jumped=1
    lấy trái 2 (idx 2), jumped=1 -> counts[2]=1
    => idx[2:4] = [3:-1, 2:2], counts = [1, 0, 1, 0]
sort(0,4): trộn [1:-1, 0:2] + [3:-1, 2:2]   (chỉ số:giá trị)
  lấy trái -1 (idx 1), jumped=0 -> counts[1]=0
  lấy phải -1 (idx 3) -> jumped=1
  lấy trái 2 (idx 0), jumped=1 -> counts[0]=2
  lấy phải 2 (idx 2) -> jumped=2
  => idx[0:4] = [1:-1, 3:-1, 0:2, 2:2], counts = [2, 0, 1, 0]
Kết quả: [2, 0, 1, 0]


### Vì sao Cách 3 đúng? (invariant bằng lời)

**Invariant của hàm `sort(lo, hi)`:** khi hàm trả về, (1) `idx[lo:hi]` là các chỉ số gốc của đoạn đó, **sắp xếp tăng dần theo giá trị**; và (2) với mỗi chỉ số `k` trong đoạn, `counts[k]` đã được cộng thêm **đúng** số phần tử `nums[m] < nums[k]` với `k < m < hi` (các số nhỏ hơn bên phải `k` **trong phạm vi đoạn này**).

- **Base case** (0–1 phần tử): không có cặp nào → đúng.
- **Bước đệ quy:** sau `sort(lo, mid)` và `sort(mid, hi)`, mỗi `k` đã có phần đóng góp từ **cùng nửa** (nhóm (a)). Với `k` thuộc nửa trái, phần còn thiếu là số phần tử nửa phải `< nums[k]` (nhóm (b)) — chính là `j − mid` lúc đưa `k` xuống, vì nửa phải đã sắp xếp và ta chỉ cho phần tử phải vượt lên khi nó **nhỏ hơn hẳn**. Với `k` thuộc nửa phải, không thiếu gì (nửa trái nằm bên trái `k`).
- Áp dụng cho `sort(0, n)`: `counts[k]` = số phần tử nhỏ hơn bên phải `k` **trong cả mảng** — đúng định nghĩa.

**Độ phức tạp** (gọi `n = len(nums)`):

- Thời gian: **`O(n log n)`** — giống merge sort (mỗi tầng trộn `O(n)`, có `≈ log2 n` tầng); việc đếm chỉ thêm `O(1)` vào mỗi bước trộn.
- Bộ nhớ: **`O(n)`** cho `idx`, `counts`, list tạm `merged`; ngăn xếp đệ quy `O(log n)`.
- Không phụ thuộc phạm vi giá trị, không cần nén tọa độ — đây là điểm mạnh so với Cách 4.

## 10. Kiến thức nền 3 — Mảng đếm (frequency array), offset cho số âm, và nén tọa độ

Quay lại khung "duyệt từ phải + túi". Một cách khác để làm túi: **mảng đếm** `cnt`, trong đó `cnt[v]` = số lần giá trị `v` đã xuất hiện trong túi.

- `thêm(x)`: `cnt[x] += 1` — `O(1)`.
- `đếm_nhỏ_hơn(x)`: `cnt[min] + ... + cnt[x − 1]` — một **tổng tiền tố (prefix sum)**.

**Vấn đề 1 — số âm.** Giá trị có thể là `-10^4`, mà chỉ số list không được âm. Tệ hơn: trong Python, `cnt[-3]` **không báo lỗi** mà lặng lẽ trỏ tới phần tử **thứ 3 từ cuối**! Cách sửa cổ điển: cộng thêm một **offset** (độ dời) `OFFSET = 10^4` để `-10^4 → 0`, `0 → 10^4`, `10^4 → 2·10^4`.

**Vấn đề 2 — tổng tiền tố chậm.** `sum(cnt[:x])` duyệt tới `2·10^4 + 1` ô mỗi lần → tổng `O(n · phạm_vi)`. Ta sẽ sửa bằng Fenwick Tree ở mục sau.

**Vấn đề 3 — nếu phạm vi giá trị rất lớn** (vd. `±2^31` như bài 327), không thể tạo mảng dài 4 tỉ ô. Nhưng ta chỉ quan tâm **thứ tự** giữa các số, không quan tâm giá trị cụ thể! **Nén tọa độ (coordinate compression):** sắp xếp các giá trị **khác nhau**, rồi thay mỗi giá trị bằng **hạng (rank)** của nó trong danh sách đó.

Ví dụ `nums = [5, 2, 6, 1, 2]`: các giá trị khác nhau đã sắp xếp `sorted(set(nums)) = [1, 2, 5, 6]` → hạng (bắt đầu từ **1**, lý do ở mục Fenwick):

| Giá trị | 1 | 2 | 5 | 6 |
|---|:---:|:---:|:---:|:---:|
| Hạng | 1 | 2 | 3 | 4 |

→ `nums` thành `[3, 2, 4, 1, 2]`. Tính chất then chốt: **`a < b` khi và chỉ khi `rank[a] < rank[b]`**, và `a == b` thì cùng hạng. Vì vậy "đếm số nhỏ hơn `x`" ⟺ "đếm số có hạng nhỏ hơn `rank[x]`". Phạm vi chỉ còn `m` = số giá trị khác nhau `<= n`. Số âm cũng tự động hết rắc rối (hạng luôn dương).

In [20]:
cnt = [0] * 5
cnt[-3] += 1                                 # tưởng là giá trị -3, thực ra là ô thứ 3 TỪ CUỐI!
print("Bẫy chỉ số âm của Python: cnt[-3] += 1 ->", cnt, "(không báo lỗi!)")

OFFSET = 10**4
for v in [-10**4, -1, 0, 10**4]:
    print(f"giá trị {v:>6} -> ô {v + OFFSET:>5} trong mảng dài {2 * OFFSET + 1}")

nums = [5, 2, 6, 1, 2]
values = sorted(set(nums))                    # các giá trị khác nhau, tăng dần
rank = {v: r + 1 for r, v in enumerate(values)}   # hạng bắt đầu từ 1
print("sorted(set(nums)) =", values)
print("rank =", rank)
print("nums sau nén =", [rank[x] for x in nums])

nums_neg = [-10**4, 3, -7, 3, 10**4]
values_neg = sorted(set(nums_neg))
print("Có số âm:", nums_neg, "->", [values_neg.index(x) + 1 for x in nums_neg])

Bẫy chỉ số âm của Python: cnt[-3] += 1 -> [0, 0, 1, 0, 0] (không báo lỗi!)
giá trị -10000 -> ô     0 trong mảng dài 20001
giá trị     -1 -> ô  9999 trong mảng dài 20001
giá trị      0 -> ô 10000 trong mảng dài 20001
giá trị  10000 -> ô 20000 trong mảng dài 20001
sorted(set(nums)) = [1, 2, 5, 6]
rank = {1: 1, 2: 2, 5: 3, 6: 4}
nums sau nén = [3, 2, 4, 1, 2]
Có số âm: [-10000, 3, -7, 3, 10000] -> [1, 3, 2, 3, 4]


## 11. Kiến thức nền 4 — Fenwick Tree (Binary Indexed Tree, BIT) từ căn bản

**Bài toán con:** có mảng `cnt[1..m]` (chỉ số từ 1), cần hai thao tác xen kẽ nhiều lần:

- `update(i, delta)`: `cnt[i] += delta`.
- `query(i)`: trả về `cnt[1] + cnt[2] + ... + cnt[i]` (tổng tiền tố). Quy ước `query(0) = 0`.

Hai cách ngây thơ đều có một thao tác chậm:

| Cách | `update` | `query` |
|---|:---:|:---:|
| Giữ `cnt` thường, query thì cộng | `O(1)` | `O(m)` |
| Giữ sẵn mảng tổng tiền tố | `O(m)` (phải sửa mọi tổng phía sau) | `O(1)` |
| **Fenwick Tree** | **`O(log m)`** | **`O(log m)`** |

**Ý tưởng Fenwick:** dùng mảng `tree[1..m]`, trong đó `tree[i]` lưu tổng của **một khối** `cnt` **kết thúc tại `i`**, có độ dài bằng `lowbit(i)` — **lũy thừa của 2 nhỏ nhất** có trong biểu diễn nhị phân của `i`:

```text
tree[i] = cnt[i − lowbit(i) + 1] + ... + cnt[i]
```

Trong Python, `lowbit(i) = i & -i` (`&` là phép **AND từng bit**; `-i` trong hệ bù hai lật mọi bit của `i` rồi cộng 1, nên `i & -i` chỉ giữ lại **bit 1 thấp nhất**). Ví dụ `6 = 110₂` → bit 1 thấp nhất là `10₂ = 2` → `6 & -6 = 2`.

| i | nhị phân | `lowbit(i) = i & -i` | `tree[i]` là tổng của |
|:---:|:---:|:---:|---|
| 1 | `0001` | 1 | `cnt[1]` |
| 2 | `0010` | 2 | `cnt[1..2]` |
| 3 | `0011` | 1 | `cnt[3]` |
| 4 | `0100` | 4 | `cnt[1..4]` |
| 5 | `0101` | 1 | `cnt[5]` |
| 6 | `0110` | 2 | `cnt[5..6]` |
| 7 | `0111` | 1 | `cnt[7]` |
| 8 | `1000` | 8 | `cnt[1..8]` |

In [21]:
for i in range(1, 9):
    low = i & -i
    print(f"i={i}: nhị phân {i:04b}, lowbit = {i} & {-i} = {low}, "
          f"tree[{i}] = tổng cnt[{i - low + 1}..{i}]")

i=1: nhị phân 0001, lowbit = 1 & -1 = 1, tree[1] = tổng cnt[1..1]
i=2: nhị phân 0010, lowbit = 2 & -2 = 2, tree[2] = tổng cnt[1..2]
i=3: nhị phân 0011, lowbit = 3 & -3 = 1, tree[3] = tổng cnt[3..3]
i=4: nhị phân 0100, lowbit = 4 & -4 = 4, tree[4] = tổng cnt[1..4]
i=5: nhị phân 0101, lowbit = 5 & -5 = 1, tree[5] = tổng cnt[5..5]
i=6: nhị phân 0110, lowbit = 6 & -6 = 2, tree[6] = tổng cnt[5..6]
i=7: nhị phân 0111, lowbit = 7 & -7 = 1, tree[7] = tổng cnt[7..7]
i=8: nhị phân 1000, lowbit = 8 & -8 = 8, tree[8] = tổng cnt[1..8]


### Hai thao tác của Fenwick: đi xuống khi `query`, đi lên khi `update`

**`query(i)`** — ghép các khối từ phải sang trái: lấy `tree[i]` (khối kết thúc tại `i`), rồi nhảy tới `i − lowbit(i)` (ngay trước khối đó), lặp tới khi `i = 0`.

```text
query(7) = tree[7] + tree[6] + tree[4]
         = cnt[7]  + cnt[5..6] + cnt[1..4]  = cnt[1..7]  ✅
đường đi: 7 -(lowbit 1)-> 6 -(lowbit 2)-> 4 -(lowbit 4)-> 0 dừng
```

**`update(i, delta)`** — cộng `delta` vào **mọi khối có chứa ô `i`**: bắt đầu từ `tree[i]`, rồi nhảy tới `i + lowbit(i)` (khối lớn hơn kế tiếp chứa `i`), lặp tới khi vượt `m`.

```text
update(3, +1) với m = 8: sửa tree[3] (cnt[3]), tree[4] (cnt[1..4]), tree[8] (cnt[1..8])
đường đi: 3 -(lowbit 1)-> 4 -(lowbit 4)-> 8 -(lowbit 8)-> 16 > 8 dừng
```

Mỗi bước nhảy, `i` mất (khi query) hoặc "dồn lên" (khi update) một bit 1 thấp nhất, nên mỗi thao tác chỉ có tối đa khoảng `log2(m) + 1` bước.

⚠️ **Vì sao chỉ số phải bắt đầu từ 1?** Vì `0 & -0 = 0`: nếu gọi `update(0, ...)`, vòng `i += i & -i` thành `i += 0` → **lặp vô hạn**. Đó là lý do hạng ở mục nén tọa độ bắt đầu từ `1`.

In [22]:
class FenwickTree:
    def __init__(self, m):
        self.m = m
        self.tree = [0] * (m + 1)            # tree[0] bỏ trống, dùng tree[1..m]

    def update(self, i, delta, verbose=False):
        path = []
        while i <= self.m:
            self.tree[i] += delta
            path.append(i)
            i += i & -i                      # nhảy LÊN khối lớn hơn chứa ô i
        if verbose:
            print(f"  update: đã sửa tree{path}")

    def query(self, i, verbose=False):       # tổng cnt[1..i]
        total, path = 0, []
        while i > 0:
            total += self.tree[i]
            path.append(i)
            i -= i & -i                      # nhảy XUỐNG khối ngay trước
        if verbose:
            print(f"  query: cộng tree{path} = {total}")
        return total

bit = FenwickTree(8)
print("update(3, +1):"); bit.update(3, 1, verbose=True)
print("update(6, +2):"); bit.update(6, 2, verbose=True)
print("query(7):");      bit.query(7, verbose=True)
print("query(5):");      bit.query(5, verbose=True)
print("query(2):");      bit.query(2, verbose=True)

# Kiểm chứng Fenwick với mảng đếm "ngây thơ" trên chuỗi thao tác ngẫu nhiên
random.seed(11)
for _ in range(300):
    m = random.randint(1, 20)
    bit = FenwickTree(m)
    naive = [0] * (m + 1)
    for _ in range(30):
        i = random.randint(1, m)
        if random.random() < 0.5:
            d = random.randint(-3, 3)
            bit.update(i, d)
            naive[i] += d
        else:
            assert bit.query(i) == sum(naive[1:i + 1])
    assert bit.query(0) == 0
print("FenwickTree khớp mảng đếm ngây thơ trên 300 chuỗi thao tác ngẫu nhiên ✅")

update(3, +1):
  update: đã sửa tree[3, 4, 8]
update(6, +2):
  update: đã sửa tree[6, 8]
query(7):
  query: cộng tree[7, 6, 4] = 3
query(5):
  query: cộng tree[5, 4] = 1
query(2):
  query: cộng tree[2] = 0
FenwickTree khớp mảng đếm ngây thơ trên 300 chuỗi thao tác ngẫu nhiên ✅


## 12. Cách 4 — Fenwick Tree + nén tọa độ, duyệt từ phải sang trái

Ghép ba mảnh ghép: **khung túi** (mục 4) + **nén tọa độ** (mục 10) + **Fenwick** (mục 11). Túi bây giờ là "mảng đếm theo hạng" được quản lý bởi BIT:

```python
rank = {v: r + 1 for r, v in enumerate(sorted(set(nums)))}   # hạng từ 1..m
bit = FenwickTree(m)
for i in range(n - 1, -1, -1):
    r = rank[nums[i]]
    counts[i] = bit.query(r - 1)    # 1) số phần tử trong túi có hạng < r  (NHỎ HƠN HẲN)
    bit.update(r, 1)                # 2) bỏ nums[i] vào túi
```

Vì sao là **`query(r − 1)`** chứ không phải `query(r)`? `query(r)` đếm các hạng `1..r`, tức là **cả các số bằng `nums[i]`** (cùng hạng `r`). Đề yêu cầu nhỏ hơn **hẳn** nên chỉ lấy hạng `1..r−1`. Khi `r = 1` (số nhỏ nhất), `query(0) = 0` — đúng: không có số nào nhỏ hơn số nhỏ nhất.

Trace trên `[5, 2, 6, 1, 2]` với `rank = {1: 1, 2: 2, 5: 3, 6: 4}`. Cột `cnt` là mảng đếm theo hạng `[cnt[1], cnt[2], cnt[3], cnt[4]]` mà BIT đang biểu diễn:

| i | `nums[i]` | r | `query(r−1)` = `counts[i]` | `cnt` sau `update(r, 1)` |
|:---:|:---:|:---:|:---:|:---|
| 4 | 2 | 2 | `query(1)` = 0 | `[0, 1, 0, 0]` |
| 3 | 1 | 1 | `query(0)` = 0 | `[1, 1, 0, 0]` |
| 2 | 6 | 4 | `query(3)` = 2 | `[1, 1, 0, 1]` |
| 1 | 2 | 2 | `query(1)` = **1** | `[1, 2, 0, 1]` |
| 0 | 5 | 3 | `query(2)` = 3 | `[1, 2, 1, 1]` |

→ `counts = [3, 1, 2, 0, 0]` ✅ (khớp Cách 2). Dòng `i = 1`: túi đã có một số `2` (hạng 2) nhưng `query(1)` chỉ đếm hạng 1 → không đếm số bằng.

In [23]:
def count_smaller_bit(nums: List[int]) -> List[int]:
    n = len(nums)
    values = sorted(set(nums))                        # nén tọa độ
    rank = {v: r + 1 for r, v in enumerate(values)}   # hạng 1..m
    bit = FenwickTree(len(values))
    counts = [0] * n
    for i in range(n - 1, -1, -1):
        r = rank[nums[i]]
        counts[i] = bit.query(r - 1)                  # hạng < r  <=>  giá trị < nums[i]
        bit.update(r, 1)
    return counts

for nums, expected in examples:
    got = count_smaller_bit(nums)
    print(nums, "->", got, "| expected:", expected)
    assert got == expected
print("[5, 2, 6, 1, 2] ->", count_smaller_bit([5, 2, 6, 1, 2]))

[5, 2, 6, 1] -> [2, 1, 1, 0] | expected: [2, 1, 1, 0]
[-1] -> [0] | expected: [0]
[-1, -1] -> [0, 0] | expected: [0, 0]
[5, 2, 6, 1, 2] -> [3, 1, 2, 0, 0]


In [24]:
def trace_bit(nums):
    n = len(nums)
    values = sorted(set(nums))
    rank = {v: r + 1 for r, v in enumerate(values)}
    m = len(values)
    print("rank =", rank)
    bit = FenwickTree(m)
    cnt = [0] * (m + 1)                               # mảng đếm "thật" chỉ để in ra cho dễ nhìn
    counts = [0] * n
    for i in range(n - 1, -1, -1):
        r = rank[nums[i]]
        counts[i] = bit.query(r - 1)
        bit.update(r, 1)
        cnt[r] += 1
        print(f"i={i}, nums[i]={nums[i]}, r={r}: query({r - 1}) = {counts[i]} -> "
              f"update({r}) -> cnt[1..{m}] = {cnt[1:]}, tree[1..{m}] = {bit.tree[1:]}")
    return counts

print("Kết quả:", trace_bit([5, 2, 6, 1, 2]))

rank = {1: 1, 2: 2, 5: 3, 6: 4}
i=4, nums[i]=2, r=2: query(1) = 0 -> update(2) -> cnt[1..4] = [0, 1, 0, 0], tree[1..4] = [0, 1, 0, 1]
i=3, nums[i]=1, r=1: query(0) = 0 -> update(1) -> cnt[1..4] = [1, 1, 0, 0], tree[1..4] = [1, 2, 0, 2]
i=2, nums[i]=6, r=4: query(3) = 2 -> update(4) -> cnt[1..4] = [1, 1, 0, 1], tree[1..4] = [1, 2, 0, 3]
i=1, nums[i]=2, r=2: query(1) = 1 -> update(2) -> cnt[1..4] = [1, 2, 0, 1], tree[1..4] = [1, 3, 0, 4]
i=0, nums[i]=5, r=3: query(2) = 3 -> update(3) -> cnt[1..4] = [1, 2, 1, 1], tree[1..4] = [1, 3, 1, 5]
Kết quả: [3, 1, 2, 0, 0]


### Độ phức tạp Cách 4 và một biến thể không cần nén

Gọi `n = len(nums)`, `m` = số giá trị khác nhau (`m <= n`).

- Nén tọa độ: `sorted(set(nums))` mất `O(n log n)`; dựng `rank` mất `O(m)`.
- Vòng lặp: `n` lần, mỗi lần một `query` và một `update` `O(log m)` → `O(n log m)`.
- Tổng: **`O(n log n)`** thời gian, **`O(n)`** bộ nhớ (`rank`, `tree`, `counts`).

**Biến thể dùng offset thay cho nén:** vì đề cho `-10^4 <= nums[i] <= 10^4`, phạm vi chỉ có `20001` giá trị. Ta có thể bỏ bước nén và dùng hạng `r = nums[i] + 10^4 + 1` (cộng `+1` để hạng bắt đầu từ 1). Khi đó độ phức tạp là `O(n log U)` với `U = 20001`. Cách này ngắn hơn nhưng **phụ thuộc constraints** — gặp bài có giá trị tới `10^9` thì phải quay lại nén tọa độ. Vì vậy nên thuộc bản nén tọa độ làm bản "tổng quát".

In [25]:
def count_smaller_bit_offset(nums: List[int]) -> List[int]:
    OFFSET = 10**4 + 1                                # -10^4 -> hạng 1, 10^4 -> hạng 20001
    bit = FenwickTree(2 * 10**4 + 1)
    counts = [0] * len(nums)
    for i in range(len(nums) - 1, -1, -1):
        r = nums[i] + OFFSET
        counts[i] = bit.query(r - 1)
        bit.update(r, 1)
    return counts

for nums, expected in examples:
    assert count_smaller_bit_offset(nums) == expected
print("[-10000, 10000, -10000, 0] ->", count_smaller_bit_offset([-10**4, 10**4, -10**4, 0]))

[-10000, 10000, -10000, 0] -> [0, 2, 0, 0]


## 13. Cách 5 (tùy chọn) — Cây nhị phân tìm kiếm (BST) tự cài, đếm kích thước cây con trái

Một lựa chọn "túi" nữa, rất trực quan: **cây nhị phân tìm kiếm (Binary Search Tree — BST)**. Mỗi **nút (node)** chứa một giá trị và hai con trỏ `left`, `right`; quy tắc: mọi giá trị ở **cây con trái** nhỏ hơn nút, mọi giá trị ở **cây con phải** lớn hơn nút. Số trùng không tạo nút mới mà tăng biến đếm `dup` của nút đó.

Ta lưu thêm `left_size` = **tổng số phần tử (tính cả trùng) trong cây con trái** của nút. Khi chèn `x` (duyệt từ phải sang trái, nên cây chứa đúng các số bên phải), ta đi từ gốc xuống và **đếm luôn** số phần tử nhỏ hơn `x`:

- `x < node.val`: rẽ **trái**; mọi thứ ta bỏ qua (nút và cây con phải) đều `>= x` → không cộng gì; nhưng cây con trái sắp có thêm `x` → `node.left_size += 1`.
- `x > node.val`: rẽ **phải**; cả cây con trái **và** nút hiện tại (`dup` bản sao) đều `< x` → cộng `node.left_size + node.dup`.
- `x == node.val`: dừng; các phần tử nhỏ hơn hẳn là cây con trái → cộng `node.left_size`; tăng `node.dup`.

Trace trên `[5, 2, 6, 1]` (chèn theo thứ tự `1, 6, 2, 5`):

| Chèn | Đường đi | Cộng | `counts` |
|:---:|---|---|:---:|
| 1 | cây rỗng → 1 làm gốc | — | 0 |
| 6 | tại 1: `6 > 1` → rẽ phải, cộng `0 + 1` | 1 | 1 |
| 2 | tại 1: rẽ phải `+1`; tại 6: `2 < 6` → rẽ trái (6.left_size = 1) | 1 | 1 |
| 5 | tại 1: rẽ phải `+1`; tại 6: rẽ trái (6.left_size = 2); tại 2: `5 > 2` → rẽ phải `+0+1` | 2 | 2 |

→ `[2, 1, 1, 0]` ✅.

⚠️ **Nhược điểm lớn:** BST thường **không tự cân bằng**. Nếu `nums` tăng dần, duyệt từ phải ta chèn các số giảm dần → cây thành một **"cái que"** lệch trái dài `n` → mỗi lần chèn `O(n)`, tổng **`O(n²)`**. Trung bình trên dữ liệu ngẫu nhiên, độ cao `O(log n)` → `O(n log n)`. Muốn đảm bảo `O(log n)` phải dùng cây **tự cân bằng** (AVL, red-black, treap) — phức tạp hơn nhiều, vì vậy trong thi đấu người ta thường chọn Fenwick hoặc merge sort.

Code dưới viết `insert` bằng **vòng lặp** `while` (không đệ quy) để tránh vượt giới hạn đệ quy khi cây lệch.

In [26]:
class BSTNode:
    def __init__(self, val):
        self.val = val
        self.dup = 1            # số lần giá trị này xuất hiện
        self.left_size = 0      # tổng số phần tử (tính cả trùng) trong cây con trái
        self.left = None
        self.right = None


def bst_insert_count(root, x):
    """Chèn x vào cây (root khác None), trả về (số phần tử < x đã có trong cây, số bước đi)."""
    node, smaller, steps = root, 0, 0
    while True:
        steps += 1
        if x == node.val:
            node.dup += 1
            return smaller + node.left_size, steps
        if x < node.val:
            node.left_size += 1                     # x sẽ nằm trong cây con trái của node
            if node.left is None:
                node.left = BSTNode(x)
                return smaller, steps
            node = node.left
        else:
            smaller += node.left_size + node.dup    # cây con trái và chính node đều < x
            if node.right is None:
                node.right = BSTNode(x)
                return smaller, steps
            node = node.right


def count_smaller_bst(nums: List[int], return_steps: bool = False):
    n = len(nums)
    counts = [0] * n
    root = BSTNode(nums[-1])                        # phần tử cuối làm gốc, counts[n-1] = 0
    total_steps = 0
    for i in range(n - 2, -1, -1):
        counts[i], steps = bst_insert_count(root, nums[i])
        total_steps += steps
    return (counts, total_steps) if return_steps else counts

for nums, expected in examples:
    got = count_smaller_bst(nums)
    print(nums, "->", got, "| expected:", expected)
    assert got == expected
print("[2, -1, 2, -1] ->", count_smaller_bst([2, -1, 2, -1]))

random.seed(5)
n = 3000
_, steps_random = count_smaller_bst([random.randint(-10**4, 10**4) for _ in range(n)], return_steps=True)
_, steps_sorted = count_smaller_bst(list(range(n)), return_steps=True)
print(f"n = {n}: tổng số bước đi trong cây — ngẫu nhiên: {steps_random:,}, tăng dần (cây lệch): {steps_sorted:,}")

[5, 2, 6, 1] -> [2, 1, 1, 0] | expected: [2, 1, 1, 0]
[-1] -> [0] | expected: [0]
[-1, -1] -> [0, 0] | expected: [0, 0]
[2, -1, 2, -1] -> [2, 0, 1, 0]


n = 3000: tổng số bước đi trong cây — ngẫu nhiên: 40,247, tăng dần (cây lệch): 4,498,500


## 14. Code nộp LeetCode — Cách 3 (merge sort trên mảng chỉ số)

Chuyển `count_smaller_merge_sort` sang đúng format LeetCode:

- Đặt trong `class Solution`, method `countSmaller(self, nums: List[int]) -> List[int]`.
- Bỏ hết `print`; giữ nguyên hàm lồng `sort(lo, hi)`.
- Một tối ưu nhỏ: đọc `nums[idx[i]]`, `nums[idx[j]]` vào biến cục bộ trong vòng lặp để giảm số lần tra list (không bắt buộc).

Bản này `O(n log n)` thời gian, `O(n)` bộ nhớ, không phụ thuộc phạm vi giá trị, xử lý đúng số trùng (ưu tiên lấy trái khi bằng) và số âm (chỉ so sánh).

In [27]:
class Solution:
    def countSmaller(self, nums: List[int]) -> List[int]:
        n = len(nums)
        counts = [0] * n
        idx = list(range(n))

        def sort(lo: int, hi: int) -> None:
            if hi - lo <= 1:
                return
            mid = (lo + hi) // 2
            sort(lo, mid)
            sort(mid, hi)
            merged = []
            i, j = lo, mid
            while i < mid and j < hi:
                if nums[idx[j]] < nums[idx[i]]:
                    merged.append(idx[j])
                    j += 1
                else:
                    counts[idx[i]] += j - mid
                    merged.append(idx[i])
                    i += 1
            while i < mid:
                counts[idx[i]] += j - mid
                merged.append(idx[i])
                i += 1
            merged.extend(idx[j:hi])
            idx[lo:hi] = merged

        sort(0, n)
        return counts


sol = Solution()
assert sol.countSmaller([5, 2, 6, 1]) == [2, 1, 1, 0]          # Example 1
assert sol.countSmaller([-1]) == [0]                            # Example 2
assert sol.countSmaller([-1, -1]) == [0, 0]                     # Example 3
assert sol.countSmaller([2, 2, 1]) == [1, 1, 0]                 # số trùng
assert sol.countSmaller([2, -1, 2, -1]) == [2, 0, 1, 0]         # trùng + âm
assert sol.countSmaller([1, 2, 3, 4]) == [0, 0, 0, 0]           # tăng dần
assert sol.countSmaller([4, 3, 2, 1]) == [3, 2, 1, 0]           # giảm dần
assert sol.countSmaller([7, 7, 7, 7]) == [0, 0, 0, 0]           # toàn bằng nhau
assert sol.countSmaller([10**4, -10**4, 0]) == [2, 0, 0]        # giá trị biên
print("class Solution (merge sort) PASS mọi test ✅")

class Solution (merge sort) PASS mọi test ✅


### Bản nộp thay thế — Cách 4 (Fenwick Tree + nén tọa độ)

Ngắn hơn và cũng `O(n log n)`. Ở đây đặt tên `SolutionBIT` để không ghi đè class trên trong notebook — **khi nộp, đổi tên thành `Solution`**. Fenwick được viết gọn ngay trong method bằng hai vòng `while`.

In [28]:
class SolutionBIT:
    def countSmaller(self, nums: List[int]) -> List[int]:
        rank = {v: r + 1 for r, v in enumerate(sorted(set(nums)))}
        m = len(rank)
        tree = [0] * (m + 1)
        counts = [0] * len(nums)
        for i in range(len(nums) - 1, -1, -1):
            r = rank[nums[i]]
            k, total = r - 1, 0                  # query(r - 1): số hạng nhỏ hơn hẳn
            while k > 0:
                total += tree[k]
                k -= k & -k
            counts[i] = total
            k = r                                # update(r, +1)
            while k <= m:
                tree[k] += 1
                k += k & -k
        return counts


sol_bit = SolutionBIT()
for nums in [[5, 2, 6, 1], [-1], [-1, -1], [2, 2, 1], [2, -1, 2, -1], [4, 3, 2, 1], [10**4, -10**4, 0]]:
    assert sol_bit.countSmaller(nums) == sol.countSmaller(nums), nums
print("SolutionBIT khớp Solution trên các test ✅")

SolutionBIT khớp Solution trên các test ✅


## 15. Test đối chiếu chéo cả 5 cách với brute force

Brute force (Cách 1) là "trọng tài" vì nó làm y hệt định nghĩa. Ta sinh hàng nghìn mảng ngẫu nhiên **nhỏ** (để brute force chạy nhanh) với `random.seed` cố định, cố tình dùng **phạm vi giá trị hẹp** (`-3..3`) để có **rất nhiều số trùng** và **số âm**, cùng một số mảng có giá trị ở biên `±10^4`.

In [29]:
methods = {
    "bag_naive": count_smaller_bag_naive,
    "bisect": count_smaller_bisect,
    "merge_sort": count_smaller_merge_sort,
    "bit": count_smaller_bit,
    "bit_offset": count_smaller_bit_offset,
    "bst": count_smaller_bst,
    "Solution": Solution().countSmaller,
    "SolutionBIT": SolutionBIT().countSmaller,
}

random.seed(2026)
num_tests = 0
for _ in range(3000):
    n = random.randint(1, 12)
    if random.random() < 0.8:
        nums = [random.randint(-3, 3) for _ in range(n)]           # nhiều số trùng
    else:
        nums = [random.choice([-10**4, 10**4, random.randint(-10**4, 10**4)]) for _ in range(n)]
    expected = count_smaller_brute(nums)
    for name, f in methods.items():
        got = f(nums[:])                                           # truyền bản sao cho an toàn
        assert got == expected, (name, nums, got, expected)
    num_tests += 1
print(f"{len(methods)} cách khớp brute force trên {num_tests} mảng ngẫu nhiên ✅")

random.seed(99)
n = 10**5
big_random = [random.randint(-10**4, 10**4) for _ in range(n)]
big_sorted = sorted(big_random)
reference = count_smaller_merge_sort(big_random)

for name in ["bisect", "merge_sort", "bit", "Solution", "SolutionBIT"]:
    f = methods[name]
    t = time.perf_counter()
    got = f(big_random)
    elapsed = time.perf_counter() - t
    assert got == reference
    print(f"{name:<12} n = {n:,} (ngẫu nhiên): {elapsed:.2f} giây")

t = time.perf_counter(); count_smaller_bit(big_sorted)
print(f"{'bit':<12} n = {n:,} (đã sắp xếp): {time.perf_counter() - t:.2f} giây  <- không bị ảnh hưởng bởi thứ tự")
print("(Brute force và BST trên mảng tăng dần bị bỏ qua vì O(n²) — quá chậm.)")

8 cách khớp brute force trên 3000 mảng ngẫu nhiên ✅


bisect       n = 100,000 (ngẫu nhiên): 0.55 giây


merge_sort   n = 100,000 (ngẫu nhiên): 0.33 giây
bit          n = 100,000 (ngẫu nhiên): 0.19 giây


Solution     n = 100,000 (ngẫu nhiên): 0.29 giây
SolutionBIT  n = 100,000 (ngẫu nhiên): 0.13 giây


bit          n = 100,000 (đã sắp xếp): 0.15 giây  <- không bị ảnh hưởng bởi thứ tự
(Brute force và BST trên mảng tăng dần bị bỏ qua vì O(n²) — quá chậm.)


## 16. Vì sao đúng — tóm tắt invariant của từng cách

| Cách | Invariant (bất biến) giữ được tại mỗi bước |
|---|---|
| 1 — Brute force | Sau vòng trong của `i`, `counts[i]` đã xét **mọi** `j > i` — đúng định nghĩa. |
| 2 — `bisect` | Trước khi xử lý `i`, `sorted_seen` chứa **đúng** các số `nums[i+1:]`, sắp xếp tăng dần → `bisect_left` = số phần tử `< nums[i]`. |
| 3 — Merge sort | Sau `sort(lo, hi)`: `idx[lo:hi]` sắp xếp theo giá trị và mỗi `counts[k]` đã đếm đúng các số nhỏ hơn bên phải `k` **trong đoạn** `[lo, hi)`. Mỗi cặp nghịch thế được đếm ở đúng một bước trộn. |
| 4 — Fenwick | Trước khi xử lý `i`, BIT biểu diễn mảng đếm theo hạng của **đúng** `nums[i+1:]`; `query(r−1)` = số phần tử có hạng `< r` ⟺ giá trị `< nums[i]` (nén tọa độ giữ nguyên thứ tự). |
| 5 — BST | Trước khi chèn `nums[i]`, cây chứa đúng `nums[i+1:]` và mọi `left_size` đều đúng; trên đường đi, phần bị "bỏ lại bên trái" là toàn bộ các phần tử `< x`. |

Hai chi tiết đúng/sai then chốt chung cho mọi cách nhanh: **(1) hỏi trước, thêm sau** (Cách 2, 4, 5) và **(2) số bằng nhau không được tính** (`bisect_left`, ưu tiên trái khi bằng, `query(r−1)`, nhánh `x == node.val` chỉ cộng `left_size`).

## 17. So sánh các cách

Gọi `n = len(nums)`, `m` = số giá trị khác nhau, `U` = độ rộng phạm vi giá trị.

| Cách | Thời gian | Bộ nhớ phụ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|
| 1 — Brute force | `O(n²)` | `O(1)` | Dễ nhất, làm trọng tài | TLE với `n = 10^5` |
| 2 — List sắp xếp + `bisect` | `O(n²)` worst (do `insort`), thực tế nhanh | `O(n)` | Code 5 dòng, dễ nhớ | Lý thuyết vẫn `O(n²)`; phỏng vấn phải giải thích |
| 3 — **Merge sort chỉ số** | **`O(n log n)`** | `O(n)` | Không phụ thuộc giá trị, kỹ thuật kinh điển đếm nghịch thế | Dễ sai ở chi tiết trộn / chỉ số |
| 4 — **Fenwick + nén tọa độ** | **`O(n log n)`** | `O(n)` (`O(U)` nếu dùng offset) | Ngắn, nhanh, mở rộng tốt (đếm trong khoảng, cập nhật online) | Phải học BIT; quên nén/offset là sai với số âm |
| 5 — BST tự cài | `O(n log n)` trung bình, `O(n²)` worst | `O(n)` | Trực quan, "đếm khi chèn" | Cây lệch với dữ liệu đã sắp xếp; cây cân bằng thì phức tạp |

**Khuyến nghị:** học kỹ **Cách 3** (hiểu bản chất "đếm nghịch thế khi trộn") và **Cách 4** (công cụ đa năng nhất). Cách 2 là phương án "chữa cháy" rất tốt khi cần code nhanh.

## 18. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `bisect_right` thay vì `bisect_left`

```python
counts[i] = bisect_right(sorted_seen, nums[i])   # SAI: đếm cả số BẰNG nums[i]
```

`bisect_right` = số phần tử `<=` x. Với `[2, 2, 1]` sẽ ra `[2, 1, 0]` thay vì `[1, 1, 0]`. **Sửa:** dùng `bisect_left` (số phần tử `<` x).

### Lỗi 2: Khi trộn, ưu tiên bên phải lúc bằng nhau

```python
if nums[idx[j]] <= nums[idx[i]]:   # SAI: số bên phải BẰNG cũng được tính là "vượt lên"
```

Với `[-1, -1]` sẽ ra `[1, 0]` thay vì `[0, 0]`. **Sửa:** chỉ cho bên phải vượt lên khi **nhỏ hơn hẳn** (`<`).

### Lỗi 3: Quên cộng cho phần còn lại của nửa trái

Khi nửa phải hết trước, các phần tử trái còn lại (những số **lớn nhất**) phải được cộng `j − mid` (lúc này bằng cả nửa phải). Nếu chỉ `merged.extend(idx[i:mid])` thì chúng bị thiếu: `[5, 2, 6, 1]` sẽ ra `[1, 1, 0, 0]`.

### Lỗi 4: Sắp xếp `nums` (hoặc dùng dict giá trị → vị trí) thay vì sắp xếp mảng chỉ số

Sau khi sắp xếp, không còn biết phần tử đến từ vị trí nào; dict thì mất vị trí khi có số trùng. **Sửa:** sắp xếp `idx`, luôn cộng vào `counts[idx[k]]`.

### Lỗi 5: Fenwick dùng `query(r)` thay vì `query(r − 1)`, hoặc hạng bắt đầu từ 0

`query(r)` đếm cả số cùng hạng (số bằng) → sai như Lỗi 1. Hạng bắt đầu từ `0` thì `update(0, ...)` **lặp vô hạn** vì `0 & -0 = 0`. **Sửa:** hạng `r + 1` khi `enumerate`, dùng `query(r − 1)`.

### Lỗi 6: Dùng giá trị âm làm chỉ số mảng đếm

`cnt[nums[i]] += 1` với `nums[i] = -3` **không báo lỗi** mà sửa ô thứ 3 từ cuối → kết quả sai lặng lẽ. **Sửa:** cộng offset hoặc nén tọa độ.

### Lỗi 7: Duyệt từ trái sang phải với khung "túi"

Túi khi đó chứa các số **bên trái** → bạn đang giải bài "đếm số nhỏ hơn **đứng trước**". **Sửa:** `range(n − 1, −1, −1)`.

Cell dưới chạy các phiên bản sai để bạn **nhìn thấy** kết quả sai.

In [30]:
def wrong_bisect_right(nums):
    counts, seen = [0] * len(nums), []
    for i in range(len(nums) - 1, -1, -1):
        counts[i] = bisect_right(seen, nums[i])      # SAI
        insort(seen, nums[i])
    return counts


def wrong_merge(nums, tie_goes_right=False, forget_leftover=False):
    n = len(nums); counts = [0] * n; idx = list(range(n))
    def sort(lo, hi):
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2; sort(lo, mid); sort(mid, hi)
        merged, i, j = [], lo, mid
        while i < mid and j < hi:
            right_first = nums[idx[j]] <= nums[idx[i]] if tie_goes_right else nums[idx[j]] < nums[idx[i]]
            if right_first:
                merged.append(idx[j]); j += 1
            else:
                counts[idx[i]] += j - mid; merged.append(idx[i]); i += 1
        if forget_leftover:
            merged.extend(idx[i:mid])                # SAI: không cộng cho phần trái còn lại
        else:
            for k in idx[i:mid]:
                counts[k] += j - mid; merged.append(k)
        merged.extend(idx[j:hi]); idx[lo:hi] = merged
    sort(0, n)
    return counts


def wrong_left_to_right(nums):
    counts, seen = [0] * len(nums), []
    for i in range(len(nums)):                       # SAI: duyệt trái -> phải
        counts[i] = bisect_left(seen, nums[i]); insort(seen, nums[i])
    return counts


print("Lỗi 1 bisect_right      [2, 2, 1]    ->", wrong_bisect_right([2, 2, 1]), "| đúng:", count_smaller_brute([2, 2, 1]))
print("Lỗi 2 bằng thì lấy phải [-1, -1]     ->", wrong_merge([-1, -1], tie_goes_right=True), "| đúng:", count_smaller_brute([-1, -1]))
print("Lỗi 3 quên phần còn lại [5, 2, 6, 1] ->", wrong_merge([5, 2, 6, 1], forget_leftover=True), "| đúng:", count_smaller_brute([5, 2, 6, 1]))
print("Lỗi 5 lowbit của 0      0 & -0       ->", 0 & -0, "-> i += 0 mãi mãi (lặp vô hạn), nên KHÔNG chạy update(0)")
cnt = [0] * 4; cnt[-1] += 1
print("Lỗi 6 chỉ số âm         cnt[-1] += 1 ->", cnt, "(sửa nhầm ô cuối, không báo lỗi)")
print("Lỗi 7 duyệt trái->phải  [5, 2, 6, 1] ->", wrong_left_to_right([5, 2, 6, 1]), "| đúng:", count_smaller_brute([5, 2, 6, 1]))

Lỗi 1 bisect_right      [2, 2, 1]    -> [2, 1, 0] | đúng: [1, 1, 0]
Lỗi 2 bằng thì lấy phải [-1, -1]     -> [1, 0] | đúng: [0, 0]
Lỗi 3 quên phần còn lại [5, 2, 6, 1] -> [1, 1, 0, 0] | đúng: [2, 1, 1, 0]
Lỗi 5 lowbit của 0      0 & -0       -> 0 -> i += 0 mãi mãi (lặp vô hạn), nên KHÔNG chạy update(0)
Lỗi 6 chỉ số âm         cnt[-1] += 1 -> [0, 0, 0, 1] (sửa nhầm ô cuối, không báo lỗi)
Lỗi 7 duyệt trái->phải  [5, 2, 6, 1] -> [0, 0, 2, 0] | đúng: [2, 1, 1, 0]


## 19. Mẹo nhận dạng pattern

Khi đề có dạng **"với mỗi phần tử, đếm số phần tử ở bên trái/bên phải thỏa một điều kiện so sánh giá trị"** (nhỏ hơn, lớn hơn, nằm trong khoảng...) và `n` tới `10^5` → nghĩ ngay tới bộ ba:

1. **Duyệt theo một chiều + "túi" hỗ trợ `thêm` và `đếm theo giá trị`** → Fenwick Tree (+ nén tọa độ nếu giá trị lớn/âm), segment tree, hoặc `SortedList`.
2. **Merge sort đếm khi trộn** — đặc biệt khi điều kiện là so sánh giữa một phần tử "trái" và một phần tử "phải" (`i < j` và `f(nums[i], nums[j])`).
3. **List sắp xếp + `bisect`** cho bản code nhanh.

Các "từ khóa" gợi ý: *inversion / nghịch thế*, *smaller after self*, *greater before*, *count pairs i < j with ...*, *rank*, *how many elements so far are less than*.

**Mẹo đổi bài:**

- "Đếm số **lớn hơn** bên phải" = `countSmaller` trên mảng **đổi dấu** `[-x for x in nums]` (vì `a > b ⟺ −a < −b`).
- "Đếm số ... bên **trái**" = chạy thuật toán trên mảng **đảo ngược** rồi đảo kết quả lại (hoặc đổi chiều duyệt).
- "Tổng số nghịch thế" = `sum(countSmaller(nums))`.

## 20. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 493 — Reverse Pairs | Đếm cặp `i < j` với `nums[i] > 2·nums[j]`: merge sort, nhưng đếm bằng hai con trỏ **riêng** trước khi trộn (vì điều kiện đếm khác điều kiện sắp xếp). |
| LeetCode 327 — Count of Range Sum | Prefix sum + đếm "số đã gặp nằm trong khoảng": merge sort / Fenwick + nén tọa độ. |
| LeetCode 1649 — Create Sorted Array through Instructions | Mỗi bước cần "số phần tử nhỏ hơn" và "lớn hơn" đã có → Fenwick y hệt Cách 4. |
| LeetCode 2179 — Count Good Triplets in an Array | Fenwick đếm "nhỏ hơn bên trái" và "lớn hơn bên phải" cho mỗi vị trí trung tâm. |
| Đếm nghịch thế (Counting Inversions) | Bài kinh điển trong sách thuật toán: chính là `sum(counts)` của bài này. |

Cell dưới minh họa các mẹo đổi bài bằng code đã có.

In [31]:
nums = [3, 7, 1, 7, 2]
smaller_after = count_smaller_bit(nums)
larger_after = count_smaller_bit([-x for x in nums])                 # đổi dấu
smaller_before = count_smaller_bit(nums[::-1])[::-1]                 # đảo ngược, rồi đảo kết quả

print("nums                 :", nums)
print("nhỏ hơn bên phải     :", smaller_after)
print("lớn hơn bên phải     :", larger_after)
print("nhỏ hơn bên trái     :", smaller_before)
print("tổng số nghịch thế   :", sum(smaller_after))

for i in range(len(nums)):                                           # kiểm chứng bằng định nghĩa
    assert larger_after[i] == sum(1 for y in nums[i + 1:] if y > nums[i])
    assert smaller_before[i] == sum(1 for y in nums[:i] if y < nums[i])
print("Các mẹo đổi bài đều đúng ✅")

nums                 : [3, 7, 1, 7, 2]
nhỏ hơn bên phải     : [2, 2, 0, 1, 0]
lớn hơn bên phải     : [2, 0, 2, 0, 0]
nhỏ hơn bên trái     : [0, 1, 0, 2, 1]
tổng số nghịch thế   : 5
Các mẹo đổi bài đều đúng ✅


## 21. Bài tập luyện tập

Làm theo thứ tự từ dễ đến khó. Các cell TODO chỉ **định nghĩa hàm**, test để ở dạng comment — làm xong thì bỏ comment để tự kiểm tra. **Gợi ý** ở mục 22, **đáp án** ở mục 23 cuối notebook.

### Bài 1 — Đoán output (không chạy code)

Cho `a = [1, 2, 2, 5]`. Đoán kết quả của:

1. `bisect_left(a, 2)` và `bisect_right(a, 2)`
2. `bisect_left(a, 3)` và `bisect_left(a, 0)`
3. `countSmaller([3, 3, 1])`
4. `7 & -7` và `12 & -12`
5. `sorted(set([4, -2, 4, 0]))` và hạng (bắt đầu từ 1) của `4`

### Bài 2 — Tính tay merge sort có đếm

Với `nums = [4, 1, 3, 2]`, hãy lập bảng giống mục 9 cho **ba bước trộn** của Cách 3: mỗi bước ghi hai nửa (chỉ số : giá trị), thứ tự các phần tử được lấy, `jumped` mỗi lần lấy trái, và `counts` sau bước. Đáp án cuối cùng phải là `countSmaller([4, 1, 3, 2])`.

Gợi ý tự kiểm tra: tính `counts` bằng định nghĩa trước, rồi xem bảng của bạn có ra cùng kết quả không.

### Bài 3 — Điền TODO: bước trộn có đếm

Hoàn thiện hàm `merge_with_count(nums, left_idx, right_idx, counts)`:

- `left_idx`, `right_idx` là hai list **chỉ số**, mỗi list đã sắp xếp tăng dần theo `nums[...]`; mọi chỉ số trong `right_idx` đứng **bên phải** mọi chỉ số trong `left_idx` trong mảng gốc.
- Trả về list chỉ số sau khi trộn (tăng dần theo giá trị) và **cộng vào `counts`** số phần tử bên phải nhỏ hơn hẳn cho mỗi chỉ số bên trái.

In [32]:
def merge_with_count(nums, left_idx, right_idx, counts):
    merged = []
    i, j = 0, 0
    # TODO 1: while cả hai bên còn phần tử:
    #         - nếu nums[right_idx[j]] < nums[left_idx[i]]: lấy bên phải
    #         - ngược lại: counts[left_idx[i]] += j, rồi lấy bên trái
    # TODO 2: phần trái còn lại: mỗi phần tử cũng được cộng j
    # TODO 3: phần phải còn lại: chép nốt
    pass

# Test sau khi làm xong:
# c = [1, 0, 1, 0]
# assert merge_with_count([5, 2, 6, 1], [1, 0], [3, 2], c) == [3, 1, 0, 2]
# assert c == [2, 1, 1, 0]
# c = [0, 0]
# assert merge_with_count([-1, -1], [0], [1], c) == [0, 1] and c == [0, 0]

### Bài 4 — Tự viết lại Cách 4 (Fenwick + nén tọa độ)

Không nhìn lại code phía trên, viết `count_smaller_bit_exercise(nums)`: tự nén tọa độ, tự cài hai thao tác Fenwick (có thể viết thẳng bằng `while` như `SolutionBIT`), duyệt từ phải sang trái.

In [33]:
def count_smaller_bit_exercise(nums):
    # TODO 1: rank = hạng (bắt đầu từ 1) của mỗi giá trị trong sorted(set(nums))
    # TODO 2: tree = [0] * (m + 1)
    # TODO 3: duyệt i từ n-1 về 0: counts[i] = query(r - 1); update(r, +1)
    raise NotImplementedError

# Test sau khi làm xong:
# assert count_smaller_bit_exercise([5, 2, 6, 1]) == [2, 1, 1, 0]
# assert count_smaller_bit_exercise([-1, -1]) == [0, 0]
# assert count_smaller_bit_exercise([2, -1, 2, -1]) == [2, 0, 1, 0]

### Bài 5 — Biến thể: đếm số lớn hơn đứng **trước**

Viết `count_greater_before(nums)` trả về list `res` với `res[i]` = số chỉ số `j < i` mà `nums[j] > nums[i]` (nhìn sang **trái**, đếm số **lớn hơn hẳn**).

Ví dụ: `count_greater_before([5, 2, 6, 1])` = `[0, 1, 0, 3]` (trước `1` có `5, 2, 6` đều lớn hơn). Yêu cầu `O(n log n)`.

In [34]:
def count_greater_before(nums):
    # TODO: Fenwick duyệt TRÁI -> PHẢI; số đã gặp lớn hơn hẳn = (đã gặp) - query(r)
    #       hoặc dùng mẹo đổi bài ở mục 19
    pass

# Test sau khi làm xong:
# assert count_greater_before([5, 2, 6, 1]) == [0, 1, 0, 3]
# assert count_greater_before([2, 2, 1]) == [0, 0, 2]
# assert count_greater_before([-1, -1]) == [0, 0]

### Bài 6 — Mini challenge: LeetCode 493 Reverse Pairs

Đếm số cặp `(i, j)` với `i < j` và **`nums[i] > 2 * nums[j]`** (gọi là "reverse pair"). `1 <= n <= 5·10^4`, giá trị trong khoảng số nguyên 32-bit.

- `reverse_pairs([1, 3, 2, 3, 1])` = `2` (cặp `(1, 4)`: `3 > 2·1` và `(3, 4)`: `3 > 2·1`).
- `reverse_pairs([2, 4, 3, 5, 1])` = `3`.

Yêu cầu `O(n log n)` bằng merge sort. Chú ý: điều kiện **đếm** (`> 2·`) khác điều kiện **sắp xếp** (`<`), nên không thể "đếm khi trộn" y hệt Cách 3.

In [35]:
def reverse_pairs(nums):
    # TODO: merge sort; trước khi trộn hai nửa đã sắp xếp L và R,
    #       dùng một con trỏ trên R để đếm, với mỗi x trong L, số y trong R mà x > 2*y
    pass

# Test sau khi làm xong:
# assert reverse_pairs([1, 3, 2, 3, 1]) == 2
# assert reverse_pairs([2, 4, 3, 5, 1]) == 3
# assert reverse_pairs([-5, -5]) == 1      # -5 > 2*(-5) = -10

## 22. Gợi ý theo 3 mức

**Bài 1**
- Mức 1: `bisect_left` = số phần tử `<` x; `bisect_right` = số phần tử `<=` x.
- Mức 2: `i & -i` giữ lại bit 1 thấp nhất: viết `7` và `12` ra nhị phân.
- Mức 3: `12 = 1100₂`; `countSmaller([3, 3, 1])`: số `3` đầu không đếm số `3` thứ hai.

**Bài 2**
- Mức 1: có 3 bước trộn: `sort(0,2)`, `sort(2,4)`, `sort(0,4)`.
- Mức 2: ở bước cuối, hai nửa đã sắp xếp là `[1:1, 0:4]` và `[3:2, 2:3]`.
- Mức 3: khi lấy `4` (idx 0) ở bước cuối, cả hai phần tử phải đã vượt lên → `jumped = 2`.

**Bài 3**
- Mức 1: giống vòng trộn của Cách 3, nhưng chỉ số `j` chạy từ `0` nên "số đã vượt lên" chính là `j`.
- Mức 2: chỉ cho bên phải đi trước khi **nhỏ hơn hẳn**.
- Mức 3: sau vòng chính: `for k in left_idx[i:]: counts[k] += j; merged.append(k)` rồi `merged.extend(right_idx[j:])`.

**Bài 4**
- Mức 1: `rank = {v: r + 1 for r, v in enumerate(sorted(set(nums)))}`.
- Mức 2: query: `while k > 0: total += tree[k]; k -= k & -k`; update: `while k <= m: tree[k] += 1; k += k & -k`.
- Mức 3: `query(r − 1)` **trước**, `update(r)` **sau**.

**Bài 5**
- Mức 1: duyệt trái → phải, túi chứa các số đứng trước.
- Mức 2: số phần tử `> x` trong túi = (số phần tử đã thêm) − (số phần tử `<= x`) = `i − query(r)`.
- Mức 3 (cách khác): `count_greater_before(nums)` = `countSmaller` trên `[-x for x in reversed(nums)]`, rồi đảo kết quả.

**Bài 6**
- Mức 1: hàm đệ quy trả về **số cặp** và sắp xếp đoạn; tổng = cặp trong trái + cặp trong phải + cặp chéo.
- Mức 2: với L, R đã sắp xếp tăng dần, khi `x` trong L tăng thì số `y` trong R thỏa `x > 2y` chỉ tăng → một con trỏ `j` chỉ tiến.
- Mức 3: `for x in L: while j < len(R) and x > 2 * R[j]: j += 1; count += j`; sau đó mới trộn (`sorted(L + R)` hoặc `merge_two_sorted`).

## 23. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

1. `bisect_left(a, 2) = 1`, `bisect_right(a, 2) = 3`.
2. `bisect_left(a, 3) = 3` (ba số `1, 2, 2` nhỏ hơn 3), `bisect_left(a, 0) = 0`.
3. `countSmaller([3, 3, 1]) = [1, 1, 0]` — hai số `3` không đếm nhau.
4. `7 = 0111₂` → `7 & -7 = 1`; `12 = 1100₂` → `12 & -12 = 4`.
5. `sorted(set([4, -2, 4, 0])) = [-2, 0, 4]` → hạng của `4` là `3`.

### Đáp án Bài 2 — `nums = [4, 1, 3, 2]`, đáp án `[3, 0, 1, 0]`

| # | Lời gọi | Trộn (chỉ số : giá trị) | Diễn biến | `counts` sau bước |
|:---:|---|---|---|---|
| 1 | `sort(0,2)` | `[0]:4` + `[1]:1` | 1 vượt lên; phải hết → lấy 4, `counts[0] += 1` | `[1, 0, 0, 0]` |
| 2 | `sort(2,4)` | `[2]:3` + `[3]:2` | 2 vượt lên; phải hết → lấy 3, `counts[2] += 1` | `[1, 0, 1, 0]` |
| 3 | `sort(0,4)` | `[1,0]:[1,4]` + `[3,2]:[2,3]` | lấy 1 (jumped 0, `+0`); 2 vượt lên; 3 vượt lên; phải hết → lấy 4, `counts[0] += 2` | `[3, 0, 1, 0]` |

Cell dưới chạy `trace_merge_sort([4, 1, 3, 2])` để đối chiếu, rồi chạy đáp án code của Bài 1, 3–6.

In [36]:
print("Đối chiếu Bài 2:", trace_merge_sort([4, 1, 3, 2]))
assert count_smaller_brute([4, 1, 3, 2]) == [3, 0, 1, 0]

# ---------- Bài 1
a = [1, 2, 2, 5]
assert (bisect_left(a, 2), bisect_right(a, 2)) == (1, 3)
assert (bisect_left(a, 3), bisect_left(a, 0)) == (3, 0)
assert Solution().countSmaller([3, 3, 1]) == [1, 1, 0]
assert (7 & -7, 12 & -12) == (1, 4)
assert sorted(set([4, -2, 4, 0])) == [-2, 0, 4]


# ---------- Bài 3
def answer_merge_with_count(nums, left_idx, right_idx, counts):
    merged = []
    i, j = 0, 0
    while i < len(left_idx) and j < len(right_idx):
        if nums[right_idx[j]] < nums[left_idx[i]]:
            merged.append(right_idx[j]); j += 1
        else:
            counts[left_idx[i]] += j
            merged.append(left_idx[i]); i += 1
    for k in left_idx[i:]:
        counts[k] += j
        merged.append(k)
    merged.extend(right_idx[j:])
    return merged

c = [1, 0, 1, 0]
assert answer_merge_with_count([5, 2, 6, 1], [1, 0], [3, 2], c) == [3, 1, 0, 2]
assert c == [2, 1, 1, 0]
c = [0, 0]
assert answer_merge_with_count([-1, -1], [0], [1], c) == [0, 1] and c == [0, 0]


# ---------- Bài 4
def answer_count_smaller_bit_exercise(nums):
    rank = {v: r + 1 for r, v in enumerate(sorted(set(nums)))}
    m = len(rank)
    tree = [0] * (m + 1)
    counts = [0] * len(nums)
    for i in range(len(nums) - 1, -1, -1):
        r = rank[nums[i]]
        k, total = r - 1, 0
        while k > 0:
            total += tree[k]; k -= k & -k
        counts[i] = total
        k = r
        while k <= m:
            tree[k] += 1; k += k & -k
    return counts

assert answer_count_smaller_bit_exercise([5, 2, 6, 1]) == [2, 1, 1, 0]
assert answer_count_smaller_bit_exercise([-1, -1]) == [0, 0]
assert answer_count_smaller_bit_exercise([2, -1, 2, -1]) == [2, 0, 1, 0]
print("Đáp án Bài 1, 3, 4 PASS ✅")

  sort(0,2): trộn [0:4] + [1:1]   (chỉ số:giá trị)
    lấy phải 1 (idx 1) -> jumped=1
    lấy trái 4 (idx 0), jumped=1 -> counts[0]=1
    => idx[0:2] = [1:1, 0:4], counts = [1, 0, 0, 0]
  sort(2,4): trộn [2:3] + [3:2]   (chỉ số:giá trị)
    lấy phải 2 (idx 3) -> jumped=1
    lấy trái 3 (idx 2), jumped=1 -> counts[2]=1
    => idx[2:4] = [3:2, 2:3], counts = [1, 0, 1, 0]
sort(0,4): trộn [1:1, 0:4] + [3:2, 2:3]   (chỉ số:giá trị)
  lấy trái 1 (idx 1), jumped=0 -> counts[1]=0
  lấy phải 2 (idx 3) -> jumped=1
  lấy phải 3 (idx 2) -> jumped=2
  lấy trái 4 (idx 0), jumped=2 -> counts[0]=3
  => idx[0:4] = [1:1, 3:2, 2:3, 0:4], counts = [3, 0, 1, 0]
Đối chiếu Bài 2: [3, 0, 1, 0]
Đáp án Bài 1, 3, 4 PASS ✅


In [37]:
# ---------- Bài 5
def answer_count_greater_before(nums):
    rank = {v: r + 1 for r, v in enumerate(sorted(set(nums)))}
    bit = FenwickTree(len(rank))
    res = []
    for i, x in enumerate(nums):                  # duyệt TRÁI -> PHẢI
        r = rank[x]
        res.append(i - bit.query(r))              # đã gặp i số, trừ đi số <= x
        bit.update(r, 1)
    return res

def answer_count_greater_before_trick(nums):      # mẹo đổi bài: đảo ngược + đổi dấu
    return Solution().countSmaller([-x for x in reversed(nums)])[::-1]

assert answer_count_greater_before([5, 2, 6, 1]) == [0, 1, 0, 3]
assert answer_count_greater_before([2, 2, 1]) == [0, 0, 2]
assert answer_count_greater_before([-1, -1]) == [0, 0]


# ---------- Bài 6
def answer_reverse_pairs(nums):
    def solve(a):                                 # trả về (a đã sắp xếp, số reverse pair trong a)
        if len(a) <= 1:
            return a[:], 0
        mid = len(a) // 2
        L, cl = solve(a[:mid])
        R, cr = solve(a[mid:])
        cross, j = 0, 0
        for x in L:                               # L tăng dần -> j chỉ tiến
            while j < len(R) and x > 2 * R[j]:
                j += 1
            cross += j
        return merge_two_sorted(L, R), cl + cr + cross
    return solve(nums)[1]

assert answer_reverse_pairs([1, 3, 2, 3, 1]) == 2
assert answer_reverse_pairs([2, 4, 3, 5, 1]) == 3
assert answer_reverse_pairs([-5, -5]) == 1

random.seed(8)
for _ in range(1000):
    nums = [random.randint(-6, 6) for _ in range(random.randint(1, 10))]
    n = len(nums)
    assert answer_count_greater_before(nums) == answer_count_greater_before_trick(nums) == \
        [sum(1 for j in range(i) if nums[j] > nums[i]) for i in range(n)]
    assert answer_reverse_pairs(nums) == sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] > 2 * nums[j])
print("Đáp án Bài 5, 6 PASS và khớp brute force trên 1000 test ngẫu nhiên ✅")

Đáp án Bài 5, 6 PASS và khớp brute force trên 1000 test ngẫu nhiên ✅


## 24. Template ngắn nên nhớ

**Merge sort trên mảng chỉ số (cách chính):**

```python
class Solution:
    def countSmaller(self, nums: List[int]) -> List[int]:
        n = len(nums)
        counts, idx = [0] * n, list(range(n))

        def sort(lo, hi):
            if hi - lo <= 1:
                return
            mid = (lo + hi) // 2
            sort(lo, mid); sort(mid, hi)
            merged, i, j = [], lo, mid
            while i < mid or j < hi:
                if j == hi or (i < mid and nums[idx[i]] <= nums[idx[j]]):
                    counts[idx[i]] += j - mid      # số phần tử phải đã "vượt lên"
                    merged.append(idx[i]); i += 1
                else:
                    merged.append(idx[j]); j += 1
            idx[lo:hi] = merged

        sort(0, n)
        return counts
```

**Fenwick + nén tọa độ (bản ngắn):**

```python
rank = {v: r + 1 for r, v in enumerate(sorted(set(nums)))}
tree = [0] * (len(rank) + 1)
for i in range(n - 1, -1, -1):
    r = rank[nums[i]]
    counts[i] = query(r - 1)     # nhỏ hơn HẲN
    update(r, 1)                 # hỏi trước, thêm sau
```

**Bản "chữa cháy":** `counts[i] = bisect_left(seen, nums[i]); insort(seen, nums[i])` (duyệt từ phải).

**Câu thần chú:** *"Duyệt từ phải, túi chứa số bên phải, hỏi trước thêm sau — hoặc merge sort chỉ số: phần tử trái nhận số phần tử phải đã vượt lên. Bằng nhau thì không đếm."*

## 25. Checklist trước khi submit

- Đếm **nhỏ hơn hẳn** chưa? (`bisect_left`, trộn ưu tiên **trái** khi bằng, `query(r − 1)`). Đã test `[-1, -1]` và `[2, 2, 1]` chưa?
- Duyệt **từ phải sang trái** (`range(n − 1, −1, −1)`) và **hỏi trước, thêm sau** chưa?
- Merge sort: sắp xếp **mảng chỉ số** `idx` (không sắp xếp `nums`, không dùng dict giá trị → vị trí) chưa?
- Merge sort: đã **cộng `j − mid` cho phần trái còn lại** khi nửa phải hết chưa? Base case `hi − lo <= 1` chưa?
- Fenwick: hạng bắt đầu từ **1** (tránh lặp vô hạn ở `update(0)`), `query(0) = 0` chưa?
- Số âm: đã nén tọa độ hoặc cộng offset (không dùng giá trị âm làm chỉ số list) chưa?
- Đã thử: 3 example của đề, mảng 1 phần tử, mảng tăng dần, giảm dần, toàn số bằng nhau, giá trị biên `±10^4` chưa?
- Nếu dùng `insort`: biết nói rõ nó `O(n)` mỗi lần (lý thuyết `O(n²)`), và cây cân bằng / Fenwick mới cho `O(log n)` thật sự chưa?